# UAV3DCrop 向け Depth Anything V2 Small Metric Depth

CPU準備・監査とColab GPU処理を分離したNotebookです。上から順に実行し、「すべて実行」は使わないでください。データ、モデル重み、GPU利用はそれぞれ明示ゲートを開いたときだけ行います。独立testはモデル選択が終わるまで読みません。

最初にREADME_ja.mdを読み、このフォルダー全体をGoogle Driveの MyDrive/DepthAnythingV2-UAV/uav_metric_finetuning に置いてください。config/verified_planも必要です。

**0. Drive接続・状態復元**は最初のコードセルです。CPU→GPU切替後はこのセルを再実行してください。データ取得や画素監査は自動で実行しません。

## 0. Drive接続・状態復元

GPU切替後にも、次のコードセルから開始してください。

In [ ]:
SESSION_READY = False
from pathlib import Path
import json, shutil, subprocess, sys
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/DepthAnythingV2-UAV')
CODE_DIR = PROJECT_DIR / 'uav_metric_finetuning'
PLAN_DIR = CODE_DIR / 'config' / 'verified_plan'
GENERATED_PLAN_DIR = PROJECT_DIR / 'work' / 'uav_metric_finetuning' / 'remote_plan_generated'
DRIVE_DATA_DIR = PROJECT_DIR / 'data'
DRIVE_RGB_ROOT = DRIVE_DATA_DIR / 'uav3dcrop_rgb'
DRIVE_DEPTH_ROOT = DRIVE_DATA_DIR / 'uav3dcrop_depth'
LOCAL_DATA_DIR = Path('/content/uav3dcrop')
LOCAL_RGB_ROOT, LOCAL_DEPTH_ROOT = LOCAL_DATA_DIR / 'rgb', LOCAL_DATA_DIR / 'depth'
RUNS_DIR = PROJECT_DIR / 'runs'
MODEL_CACHE_DIR = PROJECT_DIR / 'model_cache'
PREFLIGHT_DIR = PROJECT_DIR / 'preflight'
WORK_DIR = PROJECT_DIR / 'work' / 'uav_metric_finetuning'
for directory in (PROJECT_DIR, DRIVE_DATA_DIR, RUNS_DIR, MODEL_CACHE_DIR, PREFLIGHT_DIR, WORK_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if not (CODE_DIR / 'train.py').is_file():
    raise FileNotFoundError(f'コードをDriveへ配置してください: {CODE_DIR}')
if not (PLAN_DIR / 'plan_summary.json').is_file():
    raise FileNotFoundError(f'計画ファイルがありません: {PLAN_DIR}')
sys.path.insert(0, str(CODE_DIR))
ACTIVE_RGB_ROOT, ACTIVE_DEPTH_ROOT = DRIVE_RGB_ROOT, DRIVE_DEPTH_ROOT
print('Project:', PROJECT_DIR, '| Code:', CODE_DIR, '| Plan:', PLAN_DIR)
try:
    print('Drive reported free GiB:', round(shutil.disk_usage(PROJECT_DIR).free / 1024**3, 2))
except OSError as exc:
    print('Drive free space unavailable:', exc)
print('Colab /content free GiB:', round(shutil.disk_usage('/content').free / 1024**3, 2))

# ランタイム変更・セッション終了後も、この初期化セルから復元できます。
import importlib, runtime_state as persistent_state
importlib.reload(persistent_state)
from runtime_state import (load_settings, save_settings, validate_settings, find_report,
                           preparation_context, dataset_status)
SETTINGS_PATH = WORK_DIR / 'session_settings.json'
TRAIN_CONFIG = load_settings(SETTINGS_PATH)
globals().update(TRAIN_CONFIG)
GT_SEED, GT_MAX_PIXELS = 42, 20000
MODEL_PATHS, PREFLIGHT_REPORTS = {}, {}
RUN_DOWNLOAD = ALLOW_MODEL_DOWNLOAD = RUN_GPU_PREFLIGHT = RUN_TRAINING = False
RUN_INDEPENDENT_TEST = RUN_STAGE_TO_LOCAL = RUN_DATA_AUDIT = RUN_GT_ANALYSIS = False

def audit_context_args():
    return dict(manifests={s: PLAN_DIR / f'{s}.csv' for s in ('train','validation','independent_test')},
                rgb_root=DRIVE_RGB_ROOT, depth_root=DRIVE_DEPTH_ROOT,
                physical_splits=['train','validation'], kind='audit',
                extra_files={'plan_summary': PLAN_DIR/'plan_summary.json',
                             'existing_eval': PROJECT_DIR/'results/uav3dcrop/all_selected_images.csv'})

def gt_context_args():
    return dict(manifests={'train': PLAN_DIR/'train.csv'}, rgb_root=None,
                depth_root=DRIVE_DEPTH_ROOT, physical_splits=['train'], kind='gt',
                parameters={'seed':GT_SEED,'max_pixels_per_frame':GT_MAX_PIXELS})

def refresh_preparation_state():
    global AUDIT_OK, AUDIT_REPORT, AUDIT_PATH, GT_REPORT, GT_REPORT_PATH
    AUDIT_PATH, AUDIT_REPORT, audit_reason = find_report(WORK_DIR, 'data_audit_*.json', **audit_context_args())
    AUDIT_OK = AUDIT_REPORT is not None
    GT_REPORT_PATH, GT_REPORT, gt_reason = find_report(WORK_DIR, 'gt_distribution_train_*.json', **gt_context_args())
    print('監査:', '再利用可能' if AUDIT_OK else '未完了・再監査が必要', audit_reason)
    print('GT解析:', '再利用可能' if GT_REPORT else '未完了・再解析が必要', gt_reason)
    return AUDIT_OK, GT_REPORT is not None

plan_summary = json.loads((PLAN_DIR/'plan_summary.json').read_text(encoding='utf-8'))
refresh_preparation_state()
SESSION_READY = True
print('初期化完了。Driveの設定を復元:', SETTINGS_PATH if SETTINGS_PATH.exists() else '既定の試験設定')
print('GPU切替後は環境・設定確認・モデルcacheセルを再実行し、GPU事前計測へ進んでください。')


## 1. CPU: 環境確認

CUDAがなくても停止しません。このセルはGPUを使わず、ColabのPyTorchも更新しません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

import importlib.util
INSTALL_EXTRA_DEPS = True  # 不足時のみTrue。requirementsにTorchは含めていません。
if INSTALL_EXTRA_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE_DIR / 'requirements-colab.txt')], check=True)
missing = [m for m in ('torch','transformers','huggingface_hub','cv2','tifffile','safetensors') if importlib.util.find_spec(m) is None]
if missing:
    print('不足モジュール:', missing, '必要ならINSTALL_EXTRA_DEPS=Trueでこのセルを再実行')
else:
    import torch, transformers, huggingface_hub
    print('Python', sys.version.split()[0], '| Torch', torch.__version__, '| Torch CUDA build', torch.version.cuda)
    print('Transformers', transformers.__version__, '| HF Hub', huggingface_hub.__version__)
    print('CUDA available (information only):', torch.cuda.is_available())

In [ ]:
import torch,os,ast
assert not torch.cuda.is_available()
refresh_preparation_state()
assert AUDIT_OK and GT_REPORT is not None
EXPERIMENT_DIR=PROJECT_DIR/'experiments/metric_shape_20261013T123006472484Z'
EXPERIMENT_CODE=EXPERIMENT_DIR/'source_v4'
cpu_source="from pathlib import Path\nimport os,sys,json,hashlib,subprocess,csv,time\nfrom concurrent.futures import ThreadPoolExecutor,as_completed\nimport numpy as np,torch\n\ndef sha(p):\n    h=hashlib.sha256()\n    with Path(p).open('rb') as f:\n        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)\n    return h.hexdigest()\nassert not torch.cuda.is_available(),'CPU-only analysis: do not consume GPU'\nwork=Path(__file__).resolve().parent.parent\nsource=Path(__file__).parent\nproject=work.parent.parent\nspec=json.loads((work/'analysis_spec.json').read_text())\nspec['rgb_root']=str(project/'data/uav3dcrop_rgb');spec['depth_root']=str(project/'data/uav3dcrop_depth')\nassert spec['model_caps']=={'A':65,'B':80,'C':80}\nassert sha(spec['manifest'])==spec['validation_sha256']\nrows=list(csv.DictReader(Path(spec['manifest']).open(encoding='utf-8-sig')))\nassert len(rows)==128 and all(r['split']=='validation' for r in rows)\nweights={m:sha(Path(p)/'model.safetensors') for m,p in spec['models'].items()}\ncache=Path(spec['output'])/'prediction_cache'\ndef verify(entry):\n    m,i=entry;p=cache/f'{m}_{i:03d}.npy';ctx=json.loads(p.with_suffix('.json').read_text())\n    assert ctx['weights_sha256']==weights[m] and ctx['manifest_sha256']==spec['validation_sha256']\n    assert ctx['image_relpath']==rows[i]['image_relpath'] and ctx['image_size']==518 and ctx['model_cap']==spec['model_caps'][m] and ctx['amp_inference']==False\n    a=np.load(p);assert a.shape==(518,686) and a.dtype==np.float32 and np.isfinite(a).all() and (a>0).all()\n    return str(p),sha(p)\nverified={};start=time.monotonic()\nwith ThreadPoolExecutor(max_workers=8) as pool:\n    for future in as_completed([pool.submit(verify,(m,i)) for m in spec['models'] for i in range(128)]):\n        p,h=future.result();verified[p]=h\n        if len(verified)%64==0:print('PREDICTION_CACHE_VERIFY',len(verified),'/384',flush=True)\nwith (work/'prediction_cache_integrity.json').open('x') as f:json.dump({'models_weights_sha256':weights,'files':verified,'elapsed_seconds':time.monotonic()-start},f,indent=2)\nspecpath=work/'analysis_spec_cpu.json'\nwith specpath.open('x') as f:json.dump(spec,f,indent=2)\nscript=source/'shape_analysis_v5.py'\nsubprocess.run([sys.executable,'-u',str(script),'--spec',str(specpath),'--phase','analyze'],check=True)\nprint('STATISTICS_COMPLETE; creating representative comparison figures',flush=True)\nsubprocess.run([sys.executable,'-u',str(script),'--spec',str(specpath),'--phase','figures'],check=True)\nprint('ALL_CPU_ANALYSIS_COMPLETE',flush=True)\n"
ast.parse(cpu_source)
with (EXPERIMENT_CODE/'cpu_analysis_driver.py').open('x',encoding='utf-8') as f:f.write(cpu_source)
cpu_log=EXPERIMENT_DIR/'cpu_analysis.log'
with cpu_log.open('x') as f:
    cpu_process=subprocess.Popen([sys.executable,'-u',str(EXPERIMENT_CODE/'cpu_analysis_driver.py')],stdout=f,stderr=subprocess.STDOUT,env={**os.environ,'USE_TF':'0','PYTHONUNBUFFERED':'1','OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2'})
print('CPU分析開始 PID:',cpu_process.pid,'ログ:',cpu_log,flush=True)


In [ ]:
print('CPU分析PID:',cpu_process.pid,'終了コード:',cpu_process.poll())
print(cpu_log.read_text()[-6000:])
print('統計保存:',(EXPERIMENT_DIR/'analysis/shape_summary.json').exists(),'図の選定保存:',(EXPERIMENT_DIR/'analysis/figure_selection.json').exists())


In [ ]:
assert cpu_process.poll()==0
safe_evaluator="\"\"\"Evaluate validation with a GT mask independent of model output ceiling.\"\"\"\nimport argparse,hashlib,json\nfrom pathlib import Path\nimport torch\nfrom torch.utils.data import DataLoader\nfrom transformers import AutoModelForDepthEstimation\nfrom data import read_manifest,UAVDepthDataset\nfrom metrics import evaluate_model\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument('--model',type=Path,required=True)\n    p.add_argument('--manifest',type=Path,required=True)\n    p.add_argument('--rgb-root',type=Path,required=True)\n    p.add_argument('--depth-root',type=Path,required=True)\n    p.add_argument('--gt-max-depth',type=float,required=True)\n    p.add_argument('--image-size',type=int,default=518)\n    p.add_argument('--workers',type=int,default=2)\n    p.add_argument('--output',type=Path,required=True)\n    a=p.parse_args()\n    if a.output.exists() or a.gt_max_depth<=0:p.error('Existing output protected; GT limit must be positive')\n    rows=read_manifest(a.manifest)\n    if any(r['split']!='validation' for r in rows):p.error('This experiment evaluator accepts validation only')\n    model=AutoModelForDepthEstimation.from_pretrained(a.model,local_files_only=True)\n    cap=float(model.head.max_depth)\n    assert cap==float(model.config.max_depth)\n    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    model.to(device).eval()\n    loader=DataLoader(UAVDepthDataset(rows,a.rgb_root,a.depth_root,image_size=a.image_size,training=False),batch_size=1,num_workers=a.workers,pin_memory=device.type=='cuda')\n    result={'manifest_sha256':hashlib.sha256(a.manifest.read_bytes()).hexdigest(),'model':str(a.model),'model_output_cap_m':cap,'gt_mask_max_m':a.gt_max_depth,'split':'validation','pairs':len(rows),'device':str(device),'metrics':evaluate_model(model,loader,device,a.gt_max_depth),'prediction_rescaled':False}\n    assert model.head.max_depth==cap\n    a.output.parent.mkdir(parents=True,exist_ok=True)\n    with a.output.open('x') as f:json.dump(result,f,indent=2,allow_nan=False)\n    print(json.dumps(result,indent=2))\n\nif __name__=='__main__':main()\n"
with (EXPERIMENT_CODE/'evaluate_masked.py').open('x',encoding='utf-8') as f:f.write(safe_evaluator)
import zipfile
export_path=EXPERIMENT_DIR/'experiment2_artifacts_20261013T123006472484Z.zip'
run2=PROJECT_DIR/'runs/metric_outdoor_trial2_20261013_s42_518_cap80_gt65'
with zipfile.ZipFile(export_path,'x',compression=zipfile.ZIP_DEFLATED) as z:
    for p in EXPERIMENT_DIR.iterdir():
        if p.is_file() and p.suffix in ['.json','.log']:z.write(p,p.name)
    for p in (EXPERIMENT_DIR/'analysis').rglob('*'):
        if p.is_file() and 'prediction_cache' not in p.parts:z.write(p,p.relative_to(EXPERIMENT_DIR).as_posix())
    for p in EXPERIMENT_CODE.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts:z.write(p,'source_v4/'+p.relative_to(EXPERIMENT_CODE).as_posix())
    for p in run2.iterdir():
        if p.is_file() and p.suffix in ['.json','.jsonl']:z.write(p,'run_metadata/'+p.name)
with zipfile.ZipFile(export_path) as z:assert z.testzip() is None
print('成果物ZIP:',export_path,'bytes:',export_path.stat().st_size,'SHA256:',hashlib.sha256(export_path.read_bytes()).hexdigest())
print('代表画像:',(EXPERIMENT_DIR/'analysis/figure_selection.json').read_text())


In [ ]:
import hashlib
assert export_path.is_file()
with zipfile.ZipFile(export_path) as z: assert z.testzip() is None
print('成果物ZIP:',export_path,'bytes:',export_path.stat().st_size,'SHA256:',hashlib.sha256(export_path.read_bytes()).hexdigest())
print('図の選定:',(EXPERIMENT_DIR/'analysis/figure_selection.json').read_text())
from google.colab import files
files.download(str(export_path))


In [ ]:
from IPython.display import display,Image,Markdown
figure_dir=EXPERIMENT_DIR/'analysis/figures'
preview_files=['comparison_007_corn_nadir.png','comparison_020_corn_oblique.png','comparison_101_soybean_nadir.png','signed_delta_roi_007.png','transect_007.png']
for name in preview_files:
    p=figure_dir/name
    assert p.is_file()
    display(Markdown('### '+name))
    display(Image(filename=str(p),width=1100))


In [ ]:
border_source="from pathlib import Path\nimport json,sys,time\nimport numpy as np,tifffile,torch\nfrom shape_analysis_v5 import Stats,interp\nfrom data import read_manifest,UAVDepthDataset\nfrom transformers import AutoModelForDepthEstimation\n\nassert not torch.cuda.is_available()\nwork=Path(__file__).parent.parent\nspec=json.loads((work/'analysis_spec_cpu.json').read_text())\nrows=read_manifest(spec['manifest']);out=Path(spec['output']);cache=out/'prediction_cache'\nstats={};start=time.monotonic()\nfor i,row in enumerate(rows):\n    gt=np.asarray(tifffile.imread(Path(spec['depth_root'])/row['depth_relpath']),np.float32);H,W=gt.shape\n    valid=np.isfinite(gt)&(gt>0)&(gt<=65)\n    preds={m:np.load(cache/f'{m}_{i:03d}.npy') for m in spec['models']}\n    rng=np.random.default_rng(42+i*10000+14)\n    for axis in [0,1]:\n        hh=H-14 if axis==0 else H;ww=W-14 if axis==1 else W\n        ix=rng.choice(hh*ww,size=100000,replace=False);y=ix//ww;x=ix%ww;y1=y+(14 if axis==0 else 0);x1=x+(14 if axis==1 else 0)\n        v=valid[y,x]&valid[y1,x1];y,x,y1,x1=y[v],x[v],y1[v],x1[v]\n        t=(gt[y1,x1]-gt[y,x]).astype(float)\n        edge=np.minimum.reduce([y,x,H-1-y,W-1-x,y1,x1,H-1-y1,W-1-x1])\n        for m,pred in preds.items():\n            p0,_=interp(pred,y,x,H,W);p1,_=interp(pred,y1,x1,H,W);p=p1-p0\n            for label,mask in [('0-128px',edge<128),('128-512px',(edge>=128)&(edge<512)),('>=512px',edge>=512)]:\n                for group,select in [(label,mask),(label+'|GT<5cm',mask&(np.abs(t)<.05)),(label+'|'+row['view'],mask)]:\n                    stats.setdefault(m+'|'+group,Stats()).update(t[select],p[select])\n    if (i+1)%32==0:print('BORDER_ANALYZE',i+1,'/128',flush=True)\nresult={'sample_method':'Exactly same seed/coordinates as main stride14 analysis','zones':'minimum distance of either endpoint from original image edge','stats':{k:v.result() for k,v in stats.items()},'elapsed_seconds':time.monotonic()-start}\n# One-frame CPU diagnostic: random metric head before trial vs trained head.\nold_run=Path(spec['models']['A']).parent\nmodel=AutoModelForDepthEstimation.from_pretrained(old_run/'checkpoint_epoch_0000',local_files_only=True).cpu().eval()\ndataset=UAVDepthDataset(rows,spec['rgb_root'],spec['depth_root'],image_size=518,training=False)\nitem=dataset[7]\nwith torch.inference_mode():p0=model(pixel_values=item['pixel_values'][None]).predicted_depth[0].float().numpy()\nwith (out/'initial_A0_frame007.npy').open('xb') as f:np.save(f,p0)\np1=np.load(cache/'A_007.npy');h,w=p0.shape\nyy,xx=np.indices((h,w));ed=np.minimum.reduce([yy,xx,h-1-yy,w-1-xx])\nresult['initial_head_probe']={'validation_image':rows[7]['image_relpath'],'model_cap_m':model.head.max_depth,'native_prediction_shape':list(p0.shape),'before_min_m':float(p0.min()),'before_max_m':float(p0.max()),'before_mean_m':float(p0.mean()),'before_border_17px_mean_m':float(p0[ed<17].mean()),'before_interior_67px_mean_m':float(p0[ed>=67].mean()),'trained_border_17px_mean_m':float(p1[ed<17].mean()),'trained_interior_67px_mean_m':float(p1[ed>=67].mean()),'scope':'single frame descriptive initialization diagnostic; not an architecture causality test'}\nwith (out/'border_analysis.json').open('x') as f:json.dump(result,f,indent=2,allow_nan=False)\nprint('BORDER_ANALYSIS_COMPLETE',json.dumps(result['initial_head_probe']),flush=True)\n"
with (EXPERIMENT_CODE/'analyze_borders.py').open('x',encoding='utf-8') as f:f.write(border_source)
border_log=EXPERIMENT_DIR/'border_analysis.log'
with border_log.open('x') as f:
    border_process=subprocess.Popen([sys.executable,'-u',str(EXPERIMENT_CODE/'analyze_borders.py')],stdout=f,stderr=subprocess.STDOUT,env={**os.environ,'USE_TF':'0','OMP_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2','MKL_NUM_THREADS':'2','PYTHONUNBUFFERED':'1'})
print('画像端分析 PID:',border_process.pid)


In [ ]:
print('画像端分析 終了コード:',border_process.poll())
print(border_log.read_text()[-3500:])
if (EXPERIMENT_DIR/'analysis/border_analysis.json').exists():
    border_result=json.loads((EXPERIMENT_DIR/'analysis/border_analysis.json').read_text())
    print('BORDER_RESULT_JSON',json.dumps(border_result))


In [ ]:
import base64
assert cpu_process.poll()==0 and border_process.poll()==0
report_text="# 第2回GPU実験・局所深度差の原因分析（2026-10-13）\n\n## 結論\n\n無料Colab Tesla T4でMetric Outdoor Smallの1エポック追加学習を実行し、正常終了した。今回の本学習候補はMetric Outdoor由来を優先する。絶対深度、20cm以上の差分、28px以上の間隔でRelative由来より良い結果だった。\n\nRelative由来Aの局所MAE悪化には、画像端の偽勾配が強く寄与した。端128px以内はサンプルの10.90%だが、A−Bの局所MAE増分の82.52%を占める。内部512px以遠ではAの局所MAEはB・Cより小さい。このため、Aを画像全体で形状が悪いモデルと結論づけるのは不適切で、境界対処後の再比較が必要である。\n\nCも14pxの微細形状を十分に再現しておらず、稲の高さ精度は未検証。総合局所MAEが低いだけでは、植物の立体形状が良いとは判断できない。\n\n## 1. 実施範囲と保存先\n\nA＝Relative backbone＋新Metric neck/head、前回1epoch後。B＝既存Metric Outdoor Small（未追加学習）。C＝今回Metric Outdoor Smallを1epoch追加学習。\n\n- train384組：Day019、Day028、Day040。validation128組：Day047。\n- Driveの既存512組・1024ファイル（7,850,330,776 bytes）を再利用。4並列SSDコピー約203秒、サイズ・内容ハッシュ確認。HFから再取得なし。\n- 保存済み監査・train GT解析はmanifest・保存先・ファイル・仕様・環境一致により再利用。全画像画素監査は重複実行しなかった。\n- Day001/Day007および旧52組、independent test319組は今回の学習・分析・モデル選択に不使用。\n- 前回の研究コード、データ、モデル、評価結果は保持。修正版は実験フォルダーsource_v4へ隔離した。\n\nDrive実験フォルダー：MyDrive/DepthAnythingV2-UAV/experiments/metric_shape_20261013T123006472484Z/\n\n[Drive成果物フォルダー](https://drive.google.com/drive/folders/1bqCX9xQXgngAnyVnR15RqDFjEMC9HYKl)\n\nCのチェックポイント：MyDrive/DepthAnythingV2-UAV/runs/metric_outdoor_trial2_20261013_s42_518_cap80_gt65/\n\n## 2. モデル上限80mとGT上限65m\n\nHFのhead.forwardはsigmoid出力にhead.max_depthを掛ける。config.max_depthはhead構築時にキャッシュされる。\n\n実画像（validation Corn 7.JPG）の検証：\n\n- live modelのconfigだけ80→65へ変更：予測変化0m。\n- head.max_depthも65へ変更：予測が正確に65/80＝0.8125倍。比例式との最大誤差9.54e−7m。\n- 平均予測：80m時5.660223m、65m時4.598931m。\n- したがって、既存実装のようにconfigとheadを同時変更すると、追加学習前から予測を18.75%縮小する。configを65で保存・再構築する場合もhead上限65として構築される。\n\nCはモデル上限80mを保持し、GTの有効範囲だけ65mに設定した。Bも80mのまま。Aのモデル上限は前回の65m。全モデルの比較GTマスクは65mで一致する。Cの予測を65mでクリップしていない。\n\nsource_v4/train.pyとmemory_preflight.pyへ--model-max-depthを追加。--max-depthはGT範囲。今回の指定は--max-depth 65 --model-max-depth 80。従来の動作は元のコードで保持した。単独評価用evaluate_masked.pyは--gt-max-depthだけをGTマスクへ使い、モデル出力係数を書き換えない。旧evaluate.pyの--max-depth 65でCを評価しないこと。\n\nA対Cは初期化方式に加えてモデル出力上限も異なる。上限差単独の学習性能寄与を分離した実験ではない。B対Cには上限変更の影響が混入していない。\n\n## 3. 学習条件、数値安全性、チェックポイント\n\n初期化metric_outdoor、1epoch、518×518学習crop、batch1、蓄積4、AMP、seed42、AdamW、SiLog。encoder LR5e−6、neck/head LR5e−5、weight decay0.01、workers2、PolynomialLR(power0.9)。train/validation・変換・損失は前回と同条件。validationは縦横比を保ち518×686で推論し、元3956×5280へ復元する。\n\n最初の合成事前計測はAMP勾配オーバーフローを検出して停止。失敗ログを保持した。事前計測だけスケール適応を追加し、65536と32768で非有限勾配、16384で有限勾配と実AdamW stepを確認してpass。更新スキップを成功として扱っていない。\n\n実データ学習の初期AMPスケールは前回と同じ65536を保持。1回の更新スキップ後32768へ適応。実更新95回。スキップ時にLR schedulerを進めない既存処理を維持。SiLog Loss＝0.10020453。\n\ncheckpoint_epoch_0000／0001をDriveへ保存し、complete.jsonの重み・config・trainer state全SHA256が一致した。学習後Cを実際に再読み込みして128画像の予測生成にも成功した。optimizer／scheduler／scaler stateを保存。今回、中断resumeは行っていない。試験1epochを20epochへ強制resumeする実装は追加していない。\n\n## 4. 主比較表\n\n絶対指標と総合14px局所MAEは全有効画素／全有効ペア。全モデル2,668,757,375画素、5,320,891,492局所ペアで一致。\n\n符号一致率と回帰傾きは、全128画像・各軸・各間隔100,000候補を重複なし抽出した共通GT有効座標による統計。14pxで25,552,938ペア、非ゼロGT差分20,405,080ペア。seed＝42＋画像index×10000＋間隔。件数はサンプル件数であり全ペア件数ではない。\n\n|モデル|MAE m|RMSE m|AbsRel|深度Pearson|局所MAE m（全ペア）|符号一致（サンプル）|差分回帰傾き（サンプル）|\n|---|---:|---:|---:|---:|---:|---:|---:|\n|A|2.4897|3.9063|0.12120|0.8712|0.10545|55.92%|0.0447|\n|B|13.2730|14.5340|0.65896|0.7848|0.06561|58.33%|0.0498|\n|C|1.1324|1.7490|0.05208|0.9773|0.06800|61.82%|0.1512|\n\n一定の正の深度を予測するゼロ差分基準の全ペア局所MAE＝0.06421224m。GT差分ゼロを符号一致の分母から除くため、ゼロ差分予測の非ゼロGT符号一致率は0%、傾き0、差分相関は定義不能。この基準の低MAEを良い立体形状と解釈しない。\n\n14px差分相関：A0.03527、B0.18021、C0.32438。平均絶対差分振幅／GT振幅：A1.0663、B0.3083、C0.7679。Bは差分を大幅に小さく出すため、平坦・微小差分の多い総合MAEで有利になる。\n\n## 5. GT差分の大きさ別\n\n区間は下端含む・上端除く。差分は右方向／下方向の元画像座標で計算する。符号一致はGT差分abs>1e−6のみ、予測ゼロは不一致。回帰は予測差分をGT差分へ回帰する切片付きOLSで、予測の補正には使っていない。\n\n|GT絶対差分|サンプル件数|割合|ゼロ基準 m|A局所MAE m|B局所MAE m|C局所MAE m|\n|---|---:|---:|---:|---:|---:|---:|\n|0-5cm|18,643,474|72.96%|0.01617|0.07480|0.02266|0.03451|\n|5-10cm|3,124,866|12.23%|0.07304|0.09063|0.06963|0.07031|\n|10-20cm|1,872,762|7.33%|0.14112|0.12255|0.12858|0.10600|\n|>=20cm|1,911,836|7.48%|0.44370|0.41357|0.41684|0.35423|\n\n0〜5cmが72.96%を占め、Aの差分振幅はこの区間でGTの4.258倍。A−Bの総合局所MAE増分の95.29%がこの区間から生じた。Cもこの区間ではGT振幅の2.027倍で、BよりMAEが悪い。\n\n20cm以上ではCが最良。符号一致A71.75%、B71.32%、C77.95%。Cの平均予測絶対差分14.53cmに対しGT44.37cmで、振幅比0.327。86.72%のペアで絶対差分を過小推定している。大きい差分を捉える方向へ改善したが、高さの正確な再現には不足する。\n\n## 6. 主原因：画像端の偽勾配\n\n同じ14pxサンプルを、両端点の画像端からの最短距離で分類した。\n\n|元画像の端からの距離|ペア割合|A局所MAE m|B局所MAE m|C局所MAE m|ゼロ基準 m|\n|---|---:|---:|---:|---:|---:|\n|0-128px|10.90%|0.38929|0.08717|0.09189|0.07866|\n|128-512px|29.30%|0.09682|0.07033|0.07112|0.06898|\n|>=512px|59.79%|0.05814|0.05945|0.06220|0.05934|\n\n最外128px（10.90%のペア）がA−B悪化の82.52%を占める。AはここでGT振幅の4.353倍、差分相関0.0173、符号一致46.83%。512px以上内部ではA0.05814mがB0.05945m・C0.06220mより小さい。内部だけの結果へ主指標を置き換えていない。\n\n小差分区間と画像端区間は重複する分類なので、95.29%と82.52%の寄与率を加算しない。\n\nCorn Nadir 576.JPGのRGB・GT・予測・断面を確認した。GTがほぼ一定の道路断面に対し、Aは約29.5→20mの強い勾配を出し、道路・作物境界や画像縁に広い歪みが見える。Cは絶対値と形状がGTへ近づくが、小さい局所変動は残る。Oblique例ではCが大域的な地面の奥行き傾斜をより維持し、細かな作物の凹凸は依然平滑化されている。\n\n保存済み初期A0をCPUで再読み込みし、同じ1画像を確認：初期予測は32.4946〜32.5068m、平均32.5007m（65m×sigmoid(0)付近）。学習後の端17入力px帯平均23.4807m、67入力px以上内部17.4217m。新neck/headの1epoch後の空間的な較正の偏りと整合する。\n\n観測された主要な失敗は境界の偽勾配。ゼロpadding、学習square cropと評価landscapeの差、新neck/headの学習不足は原因候補だが、各要因を分離した反実仮想実験は行っていない。特定層だけが原因と断定しない。\n\n## 7. 解像度・GTの限界\n\n短辺3956→518、幅5280→686。元14pxは入力約1.8px、ViTの14px patchは元画像約107pxに相当する。細かな葉・土の境界の入力情報は縮小で失われる可能性がある。\n\nGTを同じ低解像度へnearest縮小し、align_corners=True bilinearで復元する対照の14px局所MAEは0.03117m、差分相関0.77472、傾き0.66885。縮小／復元だけで無視できない誤差が生じる。一方、これは理論的な下限・上限ではなく、モデル誤差の何%を解像度が説明するかは分解できない。\n\n|間隔（元px）|A局所MAE m|B局所MAE m|C局所MAE m|ゼロ基準 m（共通サンプル）|\n|---|---:|---:|---:|---:|\n|1|0.00983|0.00608|0.00728|0.00504|\n|7|0.05812|0.03575|0.03883|0.03390|\n|14|0.10558|0.06566|0.06805|0.06427|\n|28|0.18346|0.11540|0.11008|0.11651|\n|56|0.30178|0.19136|0.15870|0.20085|\n|107|0.45603|0.29560|0.20456|0.32278|\n\nCは28px以上でBを上回る。107pxではC差分相関0.78879、傾き0.64065。微細14pxの改善と、より大きい構造の改善は区別する。\n\nGT float16の画像内中央値の刻みは7.8125mm／15.625mmだった。微小差分の離散化と平坦化へ注意する。\n\n3×3中央値GTとの14px差分MAEは0.0002755mと小さい。欠損3×3近傍は612/25,552,938ペア、中央値から20cm超ずれる診断領域は3,709ペア。これらを除いてもAの局所MAEはほぼ変わらず、孤立したGTスパイク・欠損近傍が今回の主因という根拠は得られなかった。ただしMVS GTの系統的な平滑化、位置ずれ、植生の再構成誤差はこの対照で否定できない。\n\nGTはカメラ軸のZ-depthであり、世界座標の垂直な植物高ではない。植物／地面のセグメンテーションも使っていない。差分20cmが稲の高さ20cmを直接意味するわけではない。\n\n## 8. 作物・撮影方向・距離帯\n\n作物別（全画素／全14pxペア）：\n\n|作物|A RMSE m|B RMSE m|C RMSE m|A局所MAE m|B局所MAE m|C局所MAE m|\n|---|---:|---:|---:|---:|---:|---:|\n|corn|4.9581|16.5976|2.8881|0.12571|0.08680|0.08715|\n|oat|3.4187|14.4569|1.3152|0.10597|0.06639|0.06933|\n|soybean|3.6021|13.3433|0.9763|0.09378|0.05350|0.05644|\n|wheat|3.4340|13.5076|1.1012|0.09637|0.05575|0.05908|\n\n撮影方向別の絶対指標は共通画素サンプル（Nadir6,400,000画素、Oblique6,376,662画素）：\n\n|方向|A MAE m|B MAE m|C MAE m|A RMSE m|B RMSE m|C RMSE m|\n|---|---:|---:|---:|---:|---:|---:|\n|Nadir|1.8869|9.7196|0.7079|2.8812|9.8033|0.8973|\n|Oblique|3.0952|16.8414|1.5592|4.7171|18.0795|2.3084|\n\nNadir局所MAE：A0.08950、B0.03498、C0.04108m。Oblique：A0.12173、B0.09645、C0.09512m。CはOblique局所差でBも上回るが、Nadirの微細差分ではBの小振幅が総合MAEを有利にする。Nadirの絶対深度PearsonはA0.0661、B0.2500、C0.6705で、全画像Pearsonの高さだけではNadirの精度を判断できない。\n\n距離帯0〜15、15〜20、20〜30、30〜65mの14px局所MAE：A＝0.10376/0.09017/0.10036/0.17956m、B＝0.03023/0.04390/0.09367/0.18612m、C＝0.03708/0.05013/0.09096/0.17147m。Cは20m以遠でBより良いが、近距離ではBの小振幅に対してMAEが大きい。距離は各ペアの平均GT深度。作物・撮影方向・距離の効果を因果的に分離した比較ではない。\n\n## 9. 可視化・保存物・検証\n\n7例の28PNG：Corn Nadir 576.JPG、Oat Nadir、Soybean Nadir、Wheat Nadir、Corn Nadirの最大局所誤差例、Corn Oblique 385.JPG、Soybeanの最小A局所誤差例。選定規則・index・ROI・表示尺度はanalysis/figure_selection.jsonへ保存。\n\n各例でRGB／GT／A-B-C予測／絶対誤差／14px局所誤差／512px拡大／符号付き差分／断面を作成。深度は各例で共通のGT1〜99%表示尺度（範囲外の値は表示上クリップ）、誤差は共通尺度。断面の中央値オフセット除去は表示用だけで、評価値には一切使っていない。\n\n- analysis/shape_summary.json：全統計・GT対照・各画像結果・コードSHA。\n- analysis/shape_groups.csv：作物・方向・距離・区間・間隔の詳細。\n- analysis/border_analysis.json：画像端集計と初期head診断。\n- analysis/prediction_cache/：384 native float32予測とモデル／画像／manifest対応。\n- prediction_cache_integrity.json：重み対応・予測の形状／有限値検証とファイルSHA inventory。\n- gpu_phase_receipt.json、output_cap_probe.json、metric_outdoor_preflight_retry.json、trial_complete.json、各ログ。\n- source_v4/：GT／モデル上限分離、AMP診断、統計・図・安全単独評価コード。\n\nCPUで構文確認、ゼロ差分／符号／OLS／補間位置の単体確認を実施。GPUではforward/backward/AdamW、1epoch学習、checkpoint再読み込みを実行。CPUで初期A0モデル再読み込みと1画像forwardも実行。今回CPU学習を重複実行していない。\n\n初期AMP事前計測失敗は安全停止・ログ保持後に適応を確認して再計測。CPU復元後のZIP表示セルでhashlib import不足が発生したが、ZIP生成・CRC成功後の表示だけの問題であり、表示を修正しZIP生成は繰り返さなかった。\n\nDrive成果物ZIPは生成済み。IABの通常ダウンロードはHTTPエラーで、ローカルへの全ZIP取得は未完了。Colab出力で表示した図5枚はその画像データをローカルへ保存して視覚確認した。Drive上には全28図が保存されている。\n\n## 10. 時間・無料GPU・次の本学習\n\nTesla T4総VRAM14.563GiB、事前計測開始時空き14.461GiB。成功事前計測4.47秒、peak allocated634,380,800B／reserved813,694,976B。\n\n学習＋validation725.09秒（12分05秒）、起動・checkpoint保存を含む742.18秒（12分22秒）。学習peak allocated943,323,648B＝0.879GiB、reserved1,340,080,128B＝1.248GiB。3モデル384予測の生成約354秒。CPU形状統計約434秒、追加画像端集計約53秒（初期モデル読み込み・forwardは別）。\n\n有料Compute Unitsは利用可能なしの表示。数値の消費量は取得できず、0消費と確定した値は報告しない。追加購入・課金・プラン変更・利用制限回避なし。GPU成果物をDrive同期してGPUランタイムを切断・削除し、CPUのみへ変更した。CPU終了の最終確認は報告後の終了記録を参照。\n\n次の推奨：\n\n1. 本学習の優先初期化はMetric Outdoor Small、モデル上限80m・GT65m、SiLogの基準条件を維持。seed42／AdamW／LR・蓄積4を記録し、新しいfull runを作る。今回は20epochを実行していない。\n2. Aは画像端の対処後に再比較する。padding/context、学習・評価縦横比、端の損失寄与を独立に比較し、端を評価から除くことで改善扱いにしない。\n3. 高解像度または重複tileと、GT縮小／復元の補間整合性を先に比較する。新GPUで再計測し、測っていないVRAMを保証しない。\n4. 局所／複数間隔の差分損失は今後の別比較実験として検討。今回は追加していない。小GT差分の多数画素だけに支配されず、20cm以上の振幅・符号と、28/56/107pxも検証する。\n5. 稲／地面の対応GT、地面平面、カメラ姿勢・内部パラメータを用いる垂直高さ評価を用意する。今回は稲を含まない4作物で、稲の高さ精度を裏付ける実験ではない。\n6. independent testはモデル選択完了まで保持。今回のvalidationはDay047の1日で、未使用撮影日・圃場への一般化は未確認。\n\n20epochの単純時間概算は約4時間02分（起動・保存除外）。無料GPUの連続利用・同じGPUは保証されないため、同じ総epoch数・GT／モデル上限・データ・コード・パッケージでcheckpoint resumeできる運用を前提にする。試験1epochのcheckpointを20epochへ強制再開しない。\n\n## 付録：集計図\n\nanalysis/figures/overview_metrics.png に絶対誤差、差分区間、画像端、間隔別の集計図を追加。代表7画像の28図と合わせて29 PNGを保存する。入力数値と再生成コードplot_overview.pyも保存する。\n"
with (EXPERIMENT_DIR/'report_ja.md').open('x',encoding='utf-8') as f:f.write(report_text)
with (EXPERIMENT_DIR/'shape_compact.json').open('x') as f:json.dump(compact,f,indent=2)
with (EXPERIMENT_DIR/'previous_validation_summary.json').open('x') as f:json.dump(json.loads((PROJECT_DIR/'work/uav_metric_finetuning/execution_summary_20261013T113119407440Z.json').read_text()),f,indent=2)
with (EXPERIMENT_CODE/'plot_overview.py').open('x',encoding='utf-8') as f:f.write("from pathlib import Path\nimport json,argparse\nimport numpy as np\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\n\nap=argparse.ArgumentParser();ap.add_argument('--results-dir',type=Path,default=Path(__file__).parent);ap.add_argument('--output',type=Path,default=None);args=ap.parse_args()\nroot=args.results_dir\ns=json.loads((root/'shape_compact.json').read_text())\nb=json.loads((root/'border_analysis.json').read_text())\nc=json.loads((root/'gpu_phase_receipt.json').read_text())\nold=json.loads((root/'previous_validation_summary.json').read_text())\nfull={'A':old['epoch']['validation'],'B':old['metric_outdoor_original_validation']['metrics'],'C':c['epoch']['validation']}\ncolors={'A':'#e69f00','B':'#009e73','C':'#d55e00'}\nfig,ax=plt.subplots(2,2,figsize=(13,9),constrained_layout=True)\nx=np.arange(2)\nfor j,m in enumerate(['A','B','C']):\n    ax[0,0].bar(x+(j-1)*.25,[full[m]['mae_m'],full[m]['rmse_m']],width=.25,color=colors[m],label=m)\nax[0,0].set_xticks(x,['MAE','RMSE']);ax[0,0].set_title('Full validation absolute depth error');ax[0,0].set_ylabel('m');ax[0,0].legend()\nlabels=['0-5cm','5-10cm','10-20cm','>=20cm'];x=np.arange(4)\nfor j,m in enumerate(['A','B','C']):\n    ax[0,1].bar(x+(j-1)*.2,[s['s14'][f'{m}|gt_delta:{label}|s14']['delta_mae_m'] for label in labels],width=.2,color=colors[m],label=m)\nax[0,1].plot(x,[s['s14'][f'A|gt_delta:{label}|s14']['zero_delta_mae_m'] for label in labels],'k--o',label='Zero delta')\nax[0,1].set_xticks(x,labels);ax[0,1].set_title('Local dx/dy14 error by absolute GT delta');ax[0,1].set_ylabel('m');ax[0,1].legend()\nzones=['0-128px','128-512px','>=512px'];x=np.arange(3)\nfor j,m in enumerate(['A','B','C']):ax[1,0].bar(x+(j-1)*.25,[b['stats'][f'{m}|{z}']['delta_mae_m'] for z in zones],width=.25,color=colors[m],label=m)\nax[1,0].set_xticks(x,zones);ax[1,0].set_title('Local error by distance from image border');ax[1,0].set_ylabel('m');ax[1,0].legend()\nstrides=[1,7,14,28,56,107]\nfor m in ['A','B','C']:ax[1,1].plot(strides,[s['strides'][f'{m}|all|s{st}']['delta_mae_m'] for st in strides],'o-',color=colors[m],label=m)\nax[1,1].plot(strides,[s['strides'][f'A|all|s{st}']['zero_delta_mae_m'] for st in strides],'k--',label='Zero delta')\nax[1,1].set_xscale('log');ax[1,1].set_xticks(strides,[str(v) for v in strides]);ax[1,1].set_title('Dependence on original-pixel interval');ax[1,1].set_xlabel('Original-image pixels');ax[1,1].set_ylabel('m');ax[1,1].legend()\nfor a in ax.flat:a.grid(axis='y',alpha=.2)\nfig.suptitle('A: relative initialized + 1epoch; B: original Outdoor; C: Outdoor + 1epoch\\nAbsolute metrics: full valid pixels. Local metrics: identical sampled pairs.',fontsize=13)\ntarget=args.output if args.output is not None else root/'figures/overview_metrics.png'\ntarget.parent.mkdir(parents=True,exist_ok=True)\nassert not target.exists()\nfig.savefig(target,dpi=160)\nprint(target)\n")
with (EXPERIMENT_DIR/'analysis/figures/overview_metrics.png').open('xb') as f:f.write(base64.b64decode('...'))
final_export=EXPERIMENT_DIR/'experiment2_artifacts_final_20261013T123006472484Z.zip'
with zipfile.ZipFile(final_export,'x',compression=zipfile.ZIP_DEFLATED) as z:
    for p in EXPERIMENT_DIR.iterdir():
        if p.is_file() and p.suffix in ['.json','.log','.md']:z.write(p,p.name)
    for p in (EXPERIMENT_DIR/'analysis').rglob('*'):
        if p.is_file() and 'prediction_cache' not in p.parts and p.suffix!='.npy':z.write(p,p.relative_to(EXPERIMENT_DIR).as_posix())
    for p in EXPERIMENT_CODE.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts:z.write(p,'source_v4/'+p.relative_to(EXPERIMENT_CODE).as_posix())
    for p in run2.iterdir():
        if p.is_file() and p.suffix in ['.json','.jsonl']:z.write(p,'run_metadata/'+p.name)
with zipfile.ZipFile(final_export) as z:assert z.testzip() is None
print('最終報告書:',EXPERIMENT_DIR/'report_ja.md')
print('最終成果物ZIP:',final_export,'bytes:',final_export.stat().st_size,'SHA256:',hashlib.sha256(final_export.read_bytes()).hexdigest())
print('比較図PNG数:',len(list((EXPERIMENT_DIR/'analysis/figures').glob('*.png'))))
print('GPUは終了済み・現在CPU。学習・CPU分析の終了コード0。20epoch・追加損失・test・課金は実施していません。')


In [ ]:
import psutil
assert cpu_process.poll()==0 and border_process.poll()==0
assert not torch.cuda.is_available()
assert json.loads((run2/'run_status.json').read_text())['status']=='completed'
assert (EXPERIMENT_DIR/'report_ja.md').is_file() and final_export.is_file()
assert len(list((EXPERIMENT_DIR/'analysis/figures').glob('*.png')))==29
active=[(p.pid,p.info['cmdline']) for p in psutil.process_iter(['cmdline']) if any('/train.py' in s or 'cpu_analysis_driver.py' in s or 'analyze_borders.py' in s or 'shape_analysis_v5.py' in s for s in (p.info['cmdline'] or []))]
assert not active,active
with (EXPERIMENT_DIR/'closing_receipt.json').open('x') as f:json.dump({'status':'saved_ready_to_disconnect','gpu_terminated_and_cpu_only':True,'all_analysis_processes_completed':True,'trial_epochs':1,'independent_test_used':False,'compute_units_numeric_usage':'unavailable','charges_or_purchases':False,'figures':29,'final_zip_sha256':hashlib.sha256(final_export.read_bytes()).hexdigest(),'saved_utc':datetime.now(timezone.utc).isoformat()},f,indent=2)
print('最終保存PASS: C checkpoint、統計、29図、日本語報告、コード、ZIP')
print('実行中の学習・分析プロセスなし。GPU終了済み。CPUも終了します。')
from google.colab import drive
drive.flush_and_unmount()
print('最終Drive同期・アンマウント成功。CPUランタイムを終了できます。')


In [ ]:
shape=json.loads((EXPERIMENT_DIR/'analysis/shape_summary.json').read_text())
assert shape['full_zero_delta_baseline_s14']['valid_pairs']==5320891492
compact={'full_zero':shape['full_zero_delta_baseline_s14'],'s14':{k:v for k,v in shape['stats'].items() if k.endswith('|s14')},'strides':{k:v for k,v in shape['stats'].items() if '|all|' in k},'controls':shape['gt_controls'],'sampled_absolute':shape['sampled_absolute_depth_by_group'],'quantization_median_steps':sorted(set(r['median_float16_step_m'] for r in shape['float16_quantization'])),'elapsed_seconds':shape['elapsed_seconds']}
print('SHAPE_COMPACT_JSON',json.dumps(compact))


In [ ]:
import base64,io,zipfile,hashlib,os,time
refresh_preparation_state()
assert AUDIT_OK and GT_REPORT is not None, '保存済み監査の再利用条件を確認してください。重複画素監査は開始しません。'
assert dataset_status(PLAN_DIR,DRIVE_RGB_ROOT,DRIVE_DEPTH_ROOT,['train','validation'])['complete']
EXPERIMENT_STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
EXPERIMENT_DIR=PROJECT_DIR/'experiments'/f'metric_shape_{EXPERIMENT_STAMP}'
EXPERIMENT_CODE=EXPERIMENT_DIR/'source'
EXPERIMENT_CODE.mkdir(parents=True,exist_ok=False)
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='ab5e712c0c417abd30ca7c543c4fa5ce8ff4d884f0137032255140aced31d05a'
with (EXPERIMENT_DIR/'source_package.zip').open('xb') as f:f.write(blob)
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert z.testzip() is None
    for info in z.infolist():
        target=(EXPERIMENT_CODE/info.filename).resolve(); assert target.is_relative_to(EXPERIMENT_CODE.resolve())
    z.extractall(EXPERIMENT_CODE)
experiment_log=EXPERIMENT_DIR/'gpu_execution.log'
print('実験フォルダー:',EXPERIMENT_DIR,flush=True)
print('監査・GT解析再利用PASS。512組はDriveからコピーし、HF再取得なし。',flush=True)
with experiment_log.open('x') as f:
    experiment_process=subprocess.Popen([sys.executable,'-u',str(EXPERIMENT_CODE/'run_experiment2.py'),'--project',str(PROJECT_DIR),'--work',str(EXPERIMENT_DIR),'--run-trial'],stdout=f,stderr=subprocess.STDOUT,env={**os.environ,'USE_TF':'0','PYTHONUNBUFFERED':'1'})
print('第2回実験PID:',experiment_process.pid,'ログ:',experiment_log,flush=True)
with (EXPERIMENT_DIR/'launch.json').open('x') as f: json.dump({'pid':experiment_process.pid,'source_zip_sha256':hashlib.sha256(blob).hexdigest(),'audit':str(AUDIT_PATH),'gt_report':str(GT_REPORT_PATH),'source':str(EXPERIMENT_CODE),'gt_max_depth':65,'metric_model_max_depth':80,'epochs':1,'started_utc':datetime.now(timezone.utc).isoformat()},f,indent=2)


In [ ]:
print('実験PID:',experiment_process.pid,'終了コード:',experiment_process.poll())
print(experiment_log.read_text()[-16000:])
for name in ['output_cap_probe.json','trial_complete.json']:
    p=EXPERIMENT_DIR/name
    if p.exists():print(name,p.read_text()[:1800])
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.used,memory.total,utilization.gpu','--format=csv,noheader'],capture_output=True,text=True).stdout)


In [ ]:
assert experiment_process.poll()==0
run2=PROJECT_DIR/'runs/metric_outdoor_trial2_20261013_s42_518_cap80_gt65'
rows2=[json.loads(s) for s in (run2/'epoch_metrics.jsonl').read_text().splitlines() if s.strip()]
cache=EXPERIMENT_DIR/'analysis/prediction_cache'
assert len(list(cache.glob('*.npy')))==384 and len(list(cache.glob('*.json')))==384
assert (EXPERIMENT_CODE/'shape_analysis_v5.py').is_file()
gpu_receipt={'epoch':rows2[-1],'cap_probe':json.loads((EXPERIMENT_DIR/'output_cap_probe.json').read_text()),'preflight':json.loads((EXPERIMENT_DIR/'metric_outdoor_preflight_retry.json').read_text()),'trial':json.loads((EXPERIMENT_DIR/'trial_complete.json').read_text()),'prediction_cache_files':384,'inference_elapsed_seconds_from_log':354,'gpu_phase_status':'completed','compute_units':'free T4; Colab no available Compute Units; no purchases','experiment_dir':str(EXPERIMENT_DIR),'source':str(EXPERIMENT_CODE)}
with (EXPERIMENT_DIR/'gpu_phase_receipt.json').open('x') as f:json.dump(gpu_receipt,f,indent=2)
print('GPU_RECEIPT_JSON',json.dumps(gpu_receipt))
print('384予測・チェックポイント保存確認。CPU分析コードもDrive保存済み。')
from google.colab import drive
drive.flush_and_unmount()
print('Drive同期完了。GPUを終了してCPUへ切替可能。')


In [ ]:
analysis_source="\"\"\"Paired validation-only shape diagnostics. No optimizer or downloads.\n\nAbsolute metrics are reused from identical-mask full validation evaluations.\nDetailed shape metrics use deterministic shared point samples, never claimed\nas full-pixel counts. Predictions are cached at native output resolution.\n\"\"\"\nfrom pathlib import Path\nimport argparse, csv, hashlib, json, time, math\nimport numpy as np\nimport cv2, torch, tifffile\nfrom transformers import AutoModelForDepthEstimation\nfrom data import UAVDepthDataset, read_manifest\n\ndef digest(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef write_new(p,obj):\n    with Path(p).open('x',encoding='utf-8') as f: json.dump(obj,f,indent=2,allow_nan=False)\n\ndef interp(a,y,x,H,W):\n    # Same align_corners=True bilinear coordinates as metrics.evaluate_model.\n    yy=y.astype(np.float64)*(a.shape[0]-1)/(H-1)\n    xx=x.astype(np.float64)*(a.shape[1]-1)/(W-1)\n    y0=np.floor(yy).astype(int); x0=np.floor(xx).astype(int)\n    y1=np.minimum(y0+1,a.shape[0]-1); x1=np.minimum(x0+1,a.shape[1]-1)\n    wy=yy-y0; wx=xx-x0\n    v=(a[y0,x0]*(1-wx)+a[y0,x1]*wx)*(1-wy)+(a[y1,x0]*(1-wx)+a[y1,x1]*wx)*wy\n    support=np.isfinite(a[y0,x0]) & np.isfinite(a[y0,x1]) & np.isfinite(a[y1,x0]) & np.isfinite(a[y1,x1])\n    return v,support\n\nclass Stats:\n    def __init__(self): self.a=np.zeros(15,np.float64)\n    def update(self,t,p):\n        t=np.asarray(t,np.float64); p=np.asarray(p,np.float64); n=len(t)\n        if not n: return\n        at=np.abs(t); ap=np.abs(p); nz=at>1e-6\n        self.a+=np.array([n,np.abs(p-t).sum(),at.sum(),ap.sum(),t.sum(),p.sum(),(t*t).sum(),(p*p).sum(),(t*p).sum(),nz.sum(),((t[nz]*p[nz])>0).sum(),(ap>at+1e-6).sum(),(ap<at-1e-6).sum(),(ap-at).sum(),np.square(p-t).sum()])\n    def result(self):\n        n,err,zero,absp,sx,sy,sxx,syy,sxy,nz,sign,over,under,bias,se=self.a\n        if not n: return {'sampled_pairs':0}\n        vx=sxx-sx*sx/n; vy=syy-sy*sy/n; cov=sxy-sx*sy/n\n        return {'sampled_pairs':int(n),'delta_mae_m':err/n,'delta_rmse_m':math.sqrt(se/n),'zero_delta_mae_m':zero/n,'mean_abs_gt_delta_m':zero/n,'mean_abs_pred_delta_m':absp/n,'amplitude_ratio':absp/zero if zero else None,'sign_nonzero_gt_pairs':int(nz),'sign_accuracy_nonzero_gt':sign/nz if nz else None,'overestimate_fraction':over/n,'underestimate_fraction':under/n,'abs_amplitude_bias_m':bias/n,'delta_pearson':cov/math.sqrt(vx*vy) if vx>0 and vy>0 else None,'regression_slope_with_intercept':cov/vx if vx>0 else None,'regression_intercept_m':sy/n-(cov/vx)*sx/n if vx>0 else None}\n\nclass DepthStats:\n    def __init__(self): self.a=np.zeros(9,np.float64)\n    def update(self,t,p):\n        t=np.asarray(t,np.float64);p=np.asarray(p,np.float64);err=p-t\n        self.a+=np.array([len(t),np.abs(err).sum(),np.square(err).sum(),(np.abs(err)/t).sum(),t.sum(),p.sum(),(t*t).sum(),(p*p).sum(),(t*p).sum()])\n    def result(self):\n        n,ae,se,ar,sx,sy,sxx,syy,sxy=self.a\n        if not n:return {'sampled_pixels':0}\n        vx=sxx-sx*sx/n;vy=syy-sy*sy/n;cov=sxy-sx*sy/n\n        return {'sampled_pixels':int(n),'mae_m':ae/n,'rmse_m':math.sqrt(se/n),'abs_rel':ar/n,'pearson_r':cov/math.sqrt(vx*vy) if vx>0 and vy>0 else None}\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument('--spec',type=Path,required=True)\n    ap.add_argument('--phase',choices=['infer','analyze','figures'],required=True)\n    args=ap.parse_args(); spec=json.loads(args.spec.read_text()); out=Path(spec['output']); out.mkdir(parents=True,exist_ok=True)\n    rows=read_manifest(spec['manifest']); assert len(rows)==128 and all(r['split']=='validation' and r['day']=='2025/Day047' for r in rows)\n    assert not any('Day001_' in r['scene'] or 'Day007_' in r['scene'] for r in rows)\n    manifest_sha=digest(spec['manifest']); assert manifest_sha==spec['validation_sha256']\n    dataset=UAVDepthDataset(rows,spec['rgb_root'],spec['depth_root'],image_size=518,training=False)\n    cache=out/'prediction_cache'; cache.mkdir(exist_ok=True)\n    if args.phase=='infer':\n        assert torch.cuda.is_available(), 'CUDA required for inference phase'\n        torch.set_num_threads(2); started=time.monotonic()\n        for label,model_path in spec['models'].items():\n            fingerprint=digest(Path(model_path)/'model.safetensors')\n            model=AutoModelForDepthEstimation.from_pretrained(model_path,local_files_only=True).cuda().eval()\n            assert model.head.max_depth==spec['model_caps'][label]\n            for i,row in enumerate(rows):\n                target=cache/f'{label}_{i:03d}.npy'; marker=target.with_suffix('.json')\n                context={'manifest_sha256':manifest_sha,'weights_sha256':fingerprint,'image_size':518,'image_relpath':row['image_relpath'],'model_cap':model.head.max_depth,'amp_inference':False}\n                if target.exists():\n                    assert marker.exists() and json.loads(marker.read_text())==context, 'Refusing stale prediction cache'\n                    continue\n                item=dataset[i]\n                with torch.inference_mode(): prediction=model(pixel_values=item['pixel_values'][None].cuda()).predicted_depth[0].float().cpu().numpy()\n                assert np.isfinite(prediction).all() and (prediction>0).all()\n                with target.open('xb') as f: np.save(f,prediction)\n                write_new(marker,context)\n                if (i+1)%16==0: print('INFER',label,i+1,'/128 elapsed',round(time.monotonic()-started),flush=True)\n            del model; torch.cuda.empty_cache()\n        print('GPU_INFERENCE_COMPLETE',flush=True); return\n    if args.phase=='figures': return figures(spec,rows,cache,out)\n    assert not (out/'shape_summary.json').exists(), 'Analysis already exists; reuse it'\n    edges=[0,.05,.1,.2,float('inf')]; labels=['0-5cm','5-10cm','10-20cm','>=20cm']; strides=[1,7,14,28,56,107]\n    stats={}; image_results=[]; oracle_stats={}; full_zero_n=0; full_zero_sum=0.; quantization=[];depth_stats={}\n    def add(model,group,stride,t,p):\n        key=f'{model}|{group}|s{stride}'\n        stats.setdefault(key,Stats()).update(t,p)\n    start=time.monotonic()\n    for i,row in enumerate(rows):\n        gt=np.asarray(tifffile.imread(Path(spec['depth_root'])/row['depth_relpath']),np.float32); H,W=gt.shape\n        valid=np.isfinite(gt)&(gt>0)&(gt<=65)\n        # Full-pair zero baseline at stride14, in memory-bounded row chunks.\n        for axis in [0,1]:\n            s=14\n            for top in range(0,H-s if axis==0 else H,128):\n                bottom=min(top+128,H-s if axis==0 else H)\n                g0=gt[top:bottom,:] if axis==0 else gt[top:bottom,:-s]\n                g1=gt[top+s:bottom+s,:] if axis==0 else gt[top:bottom,s:]\n                v0=valid[top:bottom,:] if axis==0 else valid[top:bottom,:-s]\n                v1=valid[top+s:bottom+s,:] if axis==0 else valid[top:bottom,s:]\n                mask=v0&v1; full_zero_n+=int(mask.sum()); full_zero_sum+=float(np.abs(g1-g0)[mask].sum(dtype=np.float64))\n        predictions={m:np.load(cache/f'{m}_{i:03d}.npy') for m in spec['models']}\n        rng_pixel=np.random.default_rng(420000+i)\n        ix=rng_pixel.choice(H*W,size=100000,replace=False); py=ix//W;px=ix%W\n        pv=valid[py,px];py=py[pv];px=px[pv];target_pixel=gt[py,px]\n        for m,pred in predictions.items():\n            predicted_pixel,_=interp(pred,py,px,H,W)\n            for group in ['all',f'view:{row[\"view\"]}',f'crop:{row[\"crop\"]}']:\n                depth_stats.setdefault(f'{m}|{group}',DepthStats()).update(target_pixel,predicted_pixel)\n        low_shape=next(iter(predictions.values())).shape\n        low_gt=cv2.resize(gt,(low_shape[1],low_shape[0]),interpolation=cv2.INTER_NEAREST)\n        low_gt[(low_gt<=0)|(low_gt>65)]=np.nan\n        median=cv2.medianBlur(gt,3)\n        interior=cv2.erode(valid.astype(np.uint8),np.ones((3,3),np.uint8)).astype(bool)\n        per={m:Stats() for m in predictions}\n        q=gt[valid][::max(1,int(valid.sum()/20000))].astype(np.float16)\n        quantization.append({'image_index':i,'median_float16_step_m':float(np.median(np.nextafter(q,np.float16(np.inf)).astype(np.float32)-q.astype(np.float32)))})\n        for stride in strides:\n            rng=np.random.default_rng(42+i*10000+stride)\n            for axis in [0,1]:\n                hh=H-stride if axis==0 else H; ww=W-stride if axis==1 else W\n                # Uniform candidates without replacement, same coordinates for every model.\n                ix=rng.choice(hh*ww,size=min(100000,hh*ww),replace=False)\n                y=ix//ww; x=ix%ww; y1=y+(stride if axis==0 else 0); x1=x+(stride if axis==1 else 0)\n                good=valid[y,x]&valid[y1,x1]; y,x,y1,x1=y[good],x[good],y1[good],x1[good]\n                t=(gt[y1,x1]-gt[y,x]).astype(np.float64); mag=np.abs(t); distance=(gt[y,x]+gt[y1,x1])/2\n                for m,pred in predictions.items():\n                    p0,_=interp(pred,y,x,H,W); p1,_=interp(pred,y1,x1,H,W); p=p1-p0\n                    for group in ['all',f'crop:{row[\"crop\"]}',f'view:{row[\"view\"]}',f'axis:{axis}']:\n                        add(m,group,stride,t,p)\n                    for j,label in enumerate(labels):\n                        select=(mag>=edges[j])&(mag<edges[j+1]); add(m,f'gt_delta:{label}',stride,t[select],p[select])\n                    for lo,hi in [(0,15),(15,20),(20,30),(30,65)]:\n                        select=(distance>=lo)&(distance<hi); add(m,f'depth:{lo}-{hi}m',stride,t[select],p[select])\n                    if stride==14:\n                        per[m].update(t,p)\n                        near_invalid=~(interior[y,x]&interior[y1,x1])\n                        median_outlier=np.maximum(np.abs(gt[y,x]-median[y,x]),np.abs(gt[y1,x1]-median[y1,x1]))>.2\n                        for group,select in [('gt_support:near_invalid',near_invalid),('gt_support:interior',~near_invalid),('gt_median_outlier:>20cm',median_outlier),('gt_median_outlier:<=20cm',~median_outlier)]:\n                            add(m,group,stride,t[select],p[select])\n                o0,v0=interp(low_gt,y,x,H,W); o1,v1=interp(low_gt,y1,x1,H,W); ov=v0&v1\n                oracle_stats.setdefault(f'GT_resize_roundtrip|s{stride}',Stats()).update(t[ov],(o1-o0)[ov])\n                oracle_stats.setdefault(f'GT_median3_control|s{stride}',Stats()).update(t,(median[y1,x1]-median[y,x]).astype(np.float64))\n        image_results.append({'index':i,'image_relpath':row['image_relpath'],'crop':row['crop'],'view':row['view'],'models':{m:s.result() for m,s in per.items()},'invalid_fraction':1-float(valid.mean()),'output_shape':list(low_shape)})\n        if (i+1)%8==0: print('ANALYZE',i+1,'/128 elapsed',round(time.monotonic()-start),flush=True)\n    payload={'analysis_code_sha256':digest(__file__),'manifest_sha256':manifest_sha,'validation_images':128,'gt_max_depth_m':65,'strides_original_pixels':strides,'sample_method':'100000 candidate pairs per image/axis/stride without replacement; seed=42+image_index*10000+stride; shared GT-valid coordinates across models','bin_edges_m':[0,.05,.1,.2,None],'sign_definition':'strict positive product, excluding GT exactly zero (abs<=1e-6); predicted zero is a mismatch','regression_definition':'OLS signed predicted delta vs signed GT delta, with intercept; no calibration of predictions','full_zero_delta_baseline_s14':{'valid_pairs':full_zero_n,'mae_m':full_zero_sum/full_zero_n},'stats':{k:v.result() for k,v in stats.items()},'gt_controls':{k:v.result() for k,v in oracle_stats.items()},'float16_quantization':quantization,'images':image_results,'elapsed_seconds':time.monotonic()-start,'limitations':['GT Z-depth is camera-axis depth, not vertical crop height','No plant/ground segmentation; crop/ground ROI labels not asserted','MVS GT noise controls are diagnostic, not proof of noise-free ground truth','Detailed shape statistics are sampled; full zero baseline and existing full validation MAE separately reported']}\n    payload['sampled_absolute_depth_by_group']={k:v.result() for k,v in depth_stats.items()}\n    payload['sampled_absolute_method']='100000 uniform candidate pixels per image, seed=420000+image_index; shared GT-valid coordinates; these are sampled, distinct from full validation metrics'\n    write_new(out/'shape_summary.json',payload)\n    fields=['model','group','stride']+list(Stats().result().keys())\n    all_rows=[]\n    for key,s in payload['stats'].items():\n        m,g,st=key.split('|'); all_rows.append({'model':m,'group':g,'stride':int(st[1:]),**s})\n    fields=['model','group','stride']+list(next(r for r in all_rows if r.get('sampled_pairs',0)>0).keys())[3:]\n    with (out/'shape_groups.csv').open('x',newline='') as f:\n        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(all_rows)\n    print('SHAPE_ANALYSIS_COMPLETE',out/'shape_summary.json',flush=True)\n\ndef figures(spec,rows,cache,out):\n    import matplotlib\n    matplotlib.use('Agg')\n    import matplotlib.pyplot as plt\n    summary=json.loads((out/'shape_summary.json').read_text()); images=summary['images']\n    ranked=sorted(images,key=lambda r:r['models']['A']['delta_mae_m']-r['models']['B']['delta_mae_m'],reverse=True)\n    selected=[]\n    for crop in ['corn','oat','soybean','wheat']:\n        selected.append(next(r['index'] for r in ranked if r['crop']==crop))\n    for view in sorted({r['view'] for r in images}):\n        candidate=max((r for r in images if r['view']==view),key=lambda r:r['models']['A']['delta_mae_m'])['index']\n        if candidate not in selected: selected.append(candidate)\n    best=min(images,key=lambda r:r['models']['A']['delta_mae_m'])['index']\n    if best not in selected: selected.append(best)\n    figdir=out/'figures'; figdir.mkdir(exist_ok=True); chosen=[]\n    for i in selected:\n        row=rows[i]; rgb=cv2.cvtColor(cv2.imread(str(Path(spec['rgb_root'])/row['image_relpath'])),cv2.COLOR_BGR2RGB)\n        gt=np.asarray(tifffile.imread(Path(spec['depth_root'])/row['depth_relpath']),np.float32); H,W=gt.shape; valid=np.isfinite(gt)&(gt>0)&(gt<=65)\n        # Full-resolution interpolation, followed by display-only reduction.\n        predictions={}\n        for m in spec['models']:\n            low=torch.from_numpy(np.load(cache/f'{m}_{i:03d}.npy'))\n            predictions[m]=torch.nn.functional.interpolate(low[None,None],size=(H,W),mode='bilinear',align_corners=True)[0,0].numpy()\n        fig,ax=plt.subplots(4,4,figsize=(18,15),constrained_layout=True)\n        depth_min=float(np.quantile(gt[valid],.01)); depth_max=float(np.quantile(gt[valid],.99))\n        # Use common GT-derived scale across ALL prediction panels; clipping labelled.\n        ax[0,0].imshow(rgb); ax[0,0].set_title('RGB')\n        ax[0,1].imshow(np.where(valid,gt,np.nan),cmap='viridis',vmin=depth_min,vmax=depth_max); ax[0,1].set_title('GT Z-depth (m)')\n        ax[0,2].imshow(~valid,cmap='gray'); ax[0,2].set_title('GT invalid mask')\n        dg=gt[:,14:]-gt[:,:-14]; dv=valid[:,14:]&valid[:,:-14]\n        ax[0,3].imshow(np.where(dv,dg,np.nan),cmap='coolwarm',vmin=-.2,vmax=.2); ax[0,3].set_title('GT signed dx14 (m), clipped ±0.2')\n        scores=np.where(dv,np.abs((predictions['A'][:,14:]-predictions['A'][:,:-14])-dg)-np.abs((predictions['B'][:,14:]-predictions['B'][:,:-14])-dg),0)\n        # Fixed-grid ROI selection uses largest median error gap to avoid a single bad GT pixel.\n        candidates=[(float(np.median(scores[y:y+512,x:x+512])),y,x) for y in range(0,H-512,384) for x in range(0,W-526,384)]\n        _,yy,xx=max(candidates); roi=(slice(yy,yy+512),slice(xx,xx+512))\n        for j,m in enumerate(['A','B','C'],1):\n            p=predictions[m]; error=np.where(valid,np.abs(p-gt),np.nan)\n            de=np.where(dv,np.abs((p[:,14:]-p[:,:-14])-dg),np.nan)\n            im=ax[j,0].imshow(p,cmap='viridis',vmin=depth_min,vmax=depth_max); ax[j,0].set_title(f'{m} predicted depth; common scale')\n            im1=ax[j,1].imshow(error,cmap='magma',vmin=0,vmax=10); ax[j,1].set_title(f'{m} absolute error (m), clipped 10')\n            im2=ax[j,2].imshow(de,cmap='magma',vmin=0,vmax=.2); ax[j,2].set_title(f'{m} local dx14 error (m), clipped 0.2')\n            ax[j,3].imshow(p[roi],cmap='viridis',vmin=depth_min,vmax=depth_max); ax[j,3].set_title(f'{m} ROI y{yy}:x{xx}, 512px')\n        for a in ax.flat: a.axis('off')\n        fig.colorbar(im,ax=ax[:,0:1],shrink=.4,label='Z-depth m; GT 1–99% display range')\n        fig.colorbar(im1,ax=ax[:,1:2],shrink=.4,label='Absolute error m')\n        fig.colorbar(im2,ax=ax[:,2:3],shrink=.4,label='Local delta error m')\n        fig.suptitle(f'Validation {i}: {row[\"crop\"]} / {row[\"view\"]} / {Path(row[\"image_relpath\"]).name}; A=relative trained B=outdoor original C=outdoor trained')\n        path=figdir/f'comparison_{i:03d}_{row[\"crop\"]}_{row[\"view\"]}.png'; assert not path.exists(); fig.savefig(path,dpi=130); plt.close(fig)\n        # Separate zoom with actual RGB/GT, because no plant-ground labels are available.\n        f,a=plt.subplots(1,5,figsize=(18,4),constrained_layout=True)\n        a[0].imshow(rgb[roi]); a[0].set_title('RGB error-selected ROI')\n        for k,(label,depth) in enumerate([('GT',gt),*list(predictions.items())],1): a[k].imshow(depth[roi],cmap='viridis',vmin=depth_min,vmax=depth_max); a[k].set_title(label)\n        for z in a:z.axis('off')\n        f.savefig(figdir/f'roi_{i:03d}.png',dpi=150); plt.close(f)\n        f,a=plt.subplots(1,4,figsize=(18,4),constrained_layout=True)\n        for k,(label,p) in enumerate([('GT',gt),*list(predictions.items())]):\n            delta=p[:,14:]-p[:,:-14]\n            a[k].imshow(np.where(dv,delta,np.nan)[roi],cmap='coolwarm',vmin=-.2,vmax=.2)\n            a[k].set_title(f'{label} signed dx14, common ±0.2m');a[k].axis('off')\n        f.savefig(figdir/f'signed_delta_roi_{i:03d}.png',dpi=150);plt.close(f)\n        f,a=plt.subplots(1,2,figsize=(14,4),constrained_layout=True)\n        line_y=yy+256; xx_range=np.arange(xx,xx+512)\n        for label,p in [('GT',gt),*list(predictions.items())]:\n            z=p[line_y,xx:xx+512].astype(float);z[~valid[line_y,xx:xx+512]]=np.nan\n            a[0].plot(xx_range,z,label=label,linewidth=1)\n            a[1].plot(xx_range,z-np.nanmedian(z),label=label,linewidth=1)\n        a[0].set_title('Raw Z-depth transect (m)');a[1].set_title('Median offset removed ONLY for display (m)')\n        for axis in a:axis.legend();axis.set_xlabel('Original-image x pixel');axis.grid(alpha=.2)\n        f.savefig(figdir/f'transect_{i:03d}.png',dpi=150);plt.close(f)\n        chosen.append({'index':i,'image_relpath':row['image_relpath'],'crop':row['crop'],'view':row['view'],'roi':[yy,xx,512,512],'display_depth_min_m':depth_min,'display_depth_max_m':depth_max,'selection':'worst A minus B local MAE per crop, plus direction worst and one best-local example','figure':str(path)})\n        print('FIGURE',path,flush=True)\n    write_new(out/'figure_selection.json',chosen)\n\nif __name__=='__main__': main()\n"
analysis_path=EXPERIMENT_CODE/'shape_analysis_v5.py'
assert not analysis_path.exists()
with analysis_path.open('x',encoding='utf-8') as f:f.write(analysis_source)
import ast
ast.parse(analysis_source)
print('CPU分析コード保存:',analysis_path,'SHA256:',hashlib.sha256(analysis_source.encode()).hexdigest())


In [ ]:
assert experiment_process.poll()==1
import shutil,ast
NEW_EXPERIMENT_CODE=EXPERIMENT_DIR/'source_v4'
shutil.copytree(EXPERIMENT_CODE,NEW_EXPERIMENT_CODE,ignore=shutil.ignore_patterns('__pycache__'))
changes={'train.py':"from __future__ import annotations\r\n\r\nimport argparse\r\nimport hashlib\r\nimport json\r\nimport math\r\nimport random\r\nimport shutil\r\nimport sys\r\nimport time\r\nimport uuid\r\nfrom pathlib import Path, PurePosixPath\r\nfrom typing import Any\r\n\r\nimport cv2\r\nimport huggingface_hub\r\nimport numpy as np\r\nimport requests\r\nimport safetensors\r\nimport tifffile\r\nimport torch\r\nimport transformers\r\nfrom safetensors.torch import load_file\r\nfrom torch.optim import AdamW\r\nfrom torch.utils.data import DataLoader\r\n\r\nif __package__:\r\n    from .data import UAVDepthDataset, read_manifest, worker_seed\r\n    from .losses import silog_loss\r\n    from .metrics import evaluate_model\r\n    from .modeling import METRIC_OUTDOOR_MODEL_ID, RELATIVE_MODEL_ID, build_metric_model\r\nelse:\r\n    from data import UAVDepthDataset, read_manifest, worker_seed\r\n    from losses import silog_loss\r\n    from metrics import evaluate_model\r\n    from modeling import METRIC_OUTDOOR_MODEL_ID, RELATIVE_MODEL_ID, build_metric_model\r\n\r\n\r\ndef sha256_file(path: Path) -> str:\r\n    digest = hashlib.sha256()\r\n    with path.open(\"rb\") as stream:\r\n        for block in iter(lambda: stream.read(1024 * 1024), b\"\"):\r\n            digest.update(block)\r\n    return digest.hexdigest()\r\n\r\n\r\ndef json_safe(value: Any) -> Any:\r\n    if isinstance(value, Path):\r\n        return str(value)\r\n    if isinstance(value, dict):\r\n        return {str(key): json_safe(item) for key, item in value.items()}\r\n    if isinstance(value, (list, tuple)):\r\n        return [json_safe(item) for item in value]\r\n    return value\r\n\r\n\r\ndef write_json_atomic(path: Path, value: Any) -> None:\r\n    temporary = path.with_name(path.name + \".tmp\")\r\n    temporary.write_text(json.dumps(json_safe(value), indent=2, allow_nan=False), encoding=\"utf-8\")\r\n    temporary.replace(path)\r\n\r\n\r\ndef seed_all(seed: int) -> None:\r\n    random.seed(seed)\r\n    np.random.seed(seed)\r\n    torch.manual_seed(seed)\r\n    torch.cuda.manual_seed_all(seed)\r\n    torch.backends.cudnn.deterministic = True\r\n    torch.backends.cudnn.benchmark = False\r\n\r\n\r\ndef check_split_isolation(train_rows: list[dict[str, str]], val_rows: list[dict[str, str]]) -> None:\r\n    if not train_rows or not val_rows:\r\n        raise ValueError(\"Train and validation manifests must both contain rows\")\r\n    if any(row[\"split\"] != \"train\" for row in train_rows):\r\n        raise ValueError(\"--train-manifest contains a non-train split label\")\r\n    if any(row[\"split\"] != \"validation\" for row in val_rows):\r\n        raise ValueError(\"--val-manifest contains a non-validation split label\")\r\n    train_scenes = {row[\"scene\"] for row in train_rows}\r\n    val_scenes = {row[\"scene\"] for row in val_rows}\r\n    train_days = {row[\"day\"] for row in train_rows}\r\n    val_days = {row[\"day\"] for row in val_rows}\r\n    if train_scenes & val_scenes or train_days & val_days:\r\n        raise ValueError(\"Train/validation must be disjoint by both scene and acquisition day\")\r\n    protected = (\"2025/Day001_\", \"2025/Day007_\")\r\n    if any(row[\"scene\"].startswith(protected) for row in train_rows + val_rows):\r\n        raise ValueError(\"The historical 52-pair evaluation days are quarantined from training and validation\")\r\n    if any(row[\"split\"] == \"independent_test\" for row in train_rows + val_rows):\r\n        raise ValueError(\"Independent test rows may never be used for training or validation\")\r\n    for label, rows in ((\"train\", train_rows), (\"validation\", val_rows)):\r\n        missing = []\r\n        # Files are checked again after roots are known in main; this protects malformed paths here.\r\n        for row in rows:\r\n            image_rel = PurePosixPath(row[\"image_relpath\"])\r\n            depth_rel = PurePosixPath(row[\"depth_relpath\"])\r\n            scene = PurePosixPath(row[\"scene\"])\r\n            if image_rel.parts[:2] != scene.parts or image_rel.parent.name != \"images\":\r\n                raise ValueError(f\"RGB path does not match the declared scene: {row['scene']} / {row['image_relpath']}\")\r\n            if depth_rel.parts[:2] != (scene.parts[0], scene.name + \"_depth\") or depth_rel.stem != image_rel.stem + \"_depth\":\r\n                raise ValueError(f\"Depth path does not match the RGB/scene: {row['image_relpath']} / {row['depth_relpath']}\")\r\n            for key, path in ((\"image_relpath\", Path(row[\"image_relpath\"])), (\"depth_relpath\", Path(row[\"depth_relpath\"]))):\r\n                if path.is_absolute() or \"..\" in path.parts:\r\n                    missing.append(row[key])\r\n        if missing:\r\n            raise ValueError(f\"Unsafe relative paths in {label} manifest: {missing[:3]}\")\r\n\r\n\r\ndef verify_local_pairs(rows: list[dict[str, str]], rgb_root: Path, depth_root: Path, label: str) -> None:\r\n    missing: list[str] = []\r\n    for row in rows:\r\n        for root, key in ((rgb_root, \"image_relpath\"), (depth_root, \"depth_relpath\")):\r\n            candidate = (root / Path(row[key])).resolve()\r\n            if not candidate.is_relative_to(root.resolve()):\r\n                raise ValueError(f\"Manifest path escapes {root}: {row[key]}\")\r\n            if not candidate.is_file():\r\n                missing.append(str(candidate))\r\n    if missing:\r\n        raise FileNotFoundError(\r\n            f\"{label} data is incomplete: {len(missing)} missing files. First paths: {missing[:5]}\"\r\n        )\r\n\r\n\r\ndef validate_preflight(path: Path, args: argparse.Namespace, init_summary: dict[str, Any]) -> dict[str, Any]:\r\n    if not path.is_file():\r\n        raise FileNotFoundError(f\"GPU preflight report not found: {path}. Run memory_preflight.py first.\")\r\n    report = json.loads(path.read_text(encoding=\"utf-8\"))\r\n    checks = {\r\n        \"status\": report.get(\"status\") == \"pass\",\r\n        \"training_feasible\": report.get(\"training_feasible\") is True,\r\n        \"initialization\": report.get(\"initialization\", {}).get(\"initialization\") == args.initialization,\r\n        \"image_size\": report.get(\"image_size\") == args.image_size,\r\n        \"batch_size\": report.get(\"batch_size\") == args.batch_size,\r\n        \"amp\": report.get(\"amp\") == args.amp,\r\n        \"source\": report.get(\"initialization\", {}).get(\"source\") == init_summary.get(\"source\"),\r\n        \"source_revision\": report.get(\"initialization\", {}).get(\"source_revision\") == init_summary.get(\"source_revision\"),\r\n        \"model_max_depth\": report.get(\"initialization\", {}).get(\"max_depth_m\") == init_summary[\"max_depth_m\"],\r\n        \"gt_max_depth\": report.get(\"gt_max_depth_m\", report.get(\"initialization\", {}).get(\"max_depth_m\")) == args.max_depth,\r\n        \"gpu\": report.get(\"device\") == torch.cuda.get_device_name(0),\r\n    }\r\n    failed = [key for key, passed in checks.items() if not passed]\r\n    if failed:\r\n        raise ValueError(f\"GPU preflight does not match this training run ({', '.join(failed)}). Rerun it with identical settings/model.\")\r\n    free_bytes, total_bytes = torch.cuda.mem_get_info()\r\n    measured_reserved = int(report[\"peak_reserved_bytes_one_synthetic_step\"])\r\n    baseline_reserved = int(report.get(\"baseline_model_reserved_bytes\", 0))\r\n    additional_step_memory = max(measured_reserved - baseline_reserved, 0)\r\n    # The model itself is already resident in this process; compare only extra step memory plus headroom.\r\n    required_free = int(additional_step_memory * 1.15) + 512 * 1024**2\r\n    report[\"training_start_free_vram_bytes\"] = int(free_bytes)\r\n    report[\"training_start_total_vram_bytes\"] = int(total_bytes)\r\n    report[\"preflight_additional_step_memory_bytes\"] = additional_step_memory\r\n    report[\"required_free_vram_bytes_with_safety_margin\"] = required_free\r\n    if free_bytes < required_free:\r\n        raise MemoryError(\r\n            f\"GPU safety stop: {free_bytes / 1024**3:.2f} GiB free, but preflight peak reserved \"\r\n            f\"plus safety margin requires {required_free / 1024**3:.2f} GiB. Close other GPU jobs or reduce resolution, then rerun preflight.\"\r\n        )\r\n    return report\r\n\r\n\r\ndef checkpoint_directories(run_dir: Path) -> list[Path]:\r\n    return sorted(\r\n        [path for path in run_dir.glob(\"checkpoint_epoch_[0-9][0-9][0-9][0-9]\") if (path / \"complete.json\").is_file()],\r\n        key=lambda path: int(path.name.rsplit(\"_\", 1)[1]),\r\n    )\r\n\r\n\r\ndef save_checkpoint(\r\n    run_dir: Path,\r\n    epoch: int,\r\n    model: torch.nn.Module,\r\n    optimizer: torch.optim.Optimizer,\r\n    scheduler: Any,\r\n    scaler: Any,\r\n    best_rmse: float,\r\n    best_epoch: int | None,\r\n) -> Path:\r\n    name = f\"checkpoint_epoch_{epoch:04d}\"\r\n    final_dir = run_dir / name\r\n    staging_dir = run_dir / (\".\" + name + \".\" + uuid.uuid4().hex + \".tmp\")\r\n    if final_dir.exists() or staging_dir.exists():\r\n        raise FileExistsError(f\"Checkpoint path already exists; preserving it: {final_dir if final_dir.exists() else staging_dir}\")\r\n    staging_dir.mkdir(parents=False)\r\n    try:\r\n        model.save_pretrained(staging_dir, safe_serialization=True)\r\n        torch.save(\r\n            {\r\n                \"completed_epoch\": epoch,\r\n                \"optimizer\": optimizer.state_dict(),\r\n                \"scheduler\": scheduler.state_dict(),\r\n                \"scaler\": scaler.state_dict(),\r\n                \"best_validation_rmse_m\": best_rmse if math.isfinite(best_rmse) else None,\r\n                \"best_epoch\": best_epoch,\r\n            },\r\n            staging_dir / \"trainer_state.pt\",\r\n        )\r\n        (staging_dir / \"complete.json\").write_text(\r\n            json.dumps({\"completed_epoch\": epoch, \"file_sha256\": {\r\n                filename: sha256_file(staging_dir / filename)\r\n                for filename in (\"model.safetensors\", \"config.json\", \"trainer_state.pt\")\r\n            }}, indent=2), encoding=\"utf-8\"\r\n        )\r\n        staging_dir.replace(final_dir)\r\n    except Exception:\r\n        # Incomplete staging directories are intentionally retained for inspection.\r\n        raise\r\n    return final_dir\r\n\r\n\r\ndef prune_checkpoints(run_dir: Path, best_epoch: int | None, keep_recent: int) -> None:\r\n    if keep_recent == 0:\r\n        return  # Preserve all completed checkpoints unless retention was explicitly requested.\r\n    checkpoints = checkpoint_directories(run_dir)\r\n    keep = {path.name for path in checkpoints[-keep_recent:]}\r\n    if best_epoch is not None:\r\n        keep.add(f\"checkpoint_epoch_{best_epoch:04d}\")\r\n    for path in checkpoints:\r\n        if path.name not in keep:\r\n            shutil.rmtree(path)\r\n\r\n\r\ndef validate_resume_signature(previous: dict, current: dict) -> None:\r\n    if previous != current:\r\n        raise ValueError(\"再開設定・manifest・コードが元runと一致しません。試験から本学習へ移る場合も新しいrunを作成してください。\")\r\n\r\n\r\ndef scaled_optimizer_step(optimizer, scheduler, scaler):\r\n    previous_scale = scaler.get_scale()\r\n    scaler.step(optimizer)\r\n    scaler.update()\r\n    updated = scaler.get_scale() >= previous_scale\r\n    if updated:\r\n        scheduler.step()\r\n    return updated\r\n\r\n\r\ndef restore_checkpoint(checkpoint: Path, model, optimizer, scheduler, scaler, device):\r\n    marker = json.loads((checkpoint / \"complete.json\").read_text(encoding=\"utf-8\"))\r\n    hashes = marker.get(\"file_sha256\", {})\r\n    for filename in (\"model.safetensors\", \"config.json\", \"trainer_state.pt\"):\r\n        path = checkpoint / filename\r\n        if not path.is_file() or hashes.get(filename) != sha256_file(path):\r\n            raise ValueError(f\"checkpointの完全性を確認できません。保存ファイルは変更しません: {path}\")\r\n    # CPU staging avoids allocating a duplicate model/optimizer state on the GPU.\r\n    model.load_state_dict(load_file(str(checkpoint / \"model.safetensors\"), device=\"cpu\"), strict=True)\r\n    state = torch.load(checkpoint / \"trainer_state.pt\", map_location=\"cpu\", weights_only=False)\r\n    if state[\"completed_epoch\"] != marker[\"completed_epoch\"]:\r\n        raise ValueError(\"checkpointのepoch情報が一致しません。\")\r\n    optimizer.load_state_dict(state[\"optimizer\"])\r\n    scheduler.load_state_dict(state[\"scheduler\"])\r\n    scaler.load_state_dict(state[\"scaler\"])\r\n    return state\r\n\r\n\r\ndef truncate_epoch_log(path: Path, completed_epoch: int) -> None:\r\n    if not path.exists():\r\n        return\r\n    retained = []\r\n    for line in path.read_text(encoding=\"utf-8\").splitlines():\r\n        try:\r\n            if int(json.loads(line)[\"epoch\"]) <= completed_epoch:\r\n                retained.append(line)\r\n        except (ValueError, KeyError, json.JSONDecodeError):\r\n            continue\r\n    path.write_text(\"\\n\".join(retained) + (\"\\n\" if retained else \"\"), encoding=\"utf-8\")\r\n\r\n\r\ndef main() -> None:\r\n    parser = argparse.ArgumentParser(description=\"Fine-tune UAV3DCrop Metric Depth from one Small checkpoint.\")\r\n    parser.add_argument(\"--train-manifest\", type=Path, required=True)\r\n    parser.add_argument(\"--val-manifest\", type=Path, required=True)\r\n    parser.add_argument(\"--rgb-root\", type=Path, required=True)\r\n    parser.add_argument(\"--depth-root\", type=Path, required=True)\r\n    parser.add_argument(\"--initialization\", choices=(\"relative_encoder\", \"metric_outdoor\"), required=True)\r\n    parser.add_argument(\"--relative-model\", default=RELATIVE_MODEL_ID)\r\n    parser.add_argument(\"--metric-model\", default=METRIC_OUTDOOR_MODEL_ID)\r\n    parser.add_argument(\"--max-depth\", type=float, required=True, help=\"Metres; derive using train-only analyze_gt.py\")\r\n    parser.add_argument(\"--model-max-depth\", type=float, default=None, help=\"Model output ceiling; --max-depth remains the GT mask limit\")\r\n    parser.add_argument(\"--image-size\", type=int, default=518)\r\n    parser.add_argument(\"--batch-size\", type=int, default=1)\r\n    parser.add_argument(\"--grad-accumulation\", type=int, default=4)\r\n    parser.add_argument(\"--epochs\", type=int, default=1)\r\n    parser.add_argument(\"--run-kind\", choices=(\"trial\", \"full\"), default=\"trial\")\r\n    parser.add_argument(\"--encoder-lr\", type=float, default=5e-6)\r\n    parser.add_argument(\"--weight-decay\", type=float, default=0.01)\r\n    parser.add_argument(\"--workers\", type=int, default=2)\r\n    parser.add_argument(\"--seed\", type=int, default=42)\r\n    parser.add_argument(\"--amp\", action=argparse.BooleanOptionalAction, default=True)\r\n    parser.add_argument(\"--save-every-epochs\", type=int, default=1)\r\n    parser.add_argument(\"--keep-recent-checkpoints\", type=int, default=0, help=\"0 preserves all; positive retention is explicit\")\r\n    parser.add_argument(\"--memory-preflight-report\", type=Path, required=True)\r\n    parser.add_argument(\"--output-dir\", type=Path, required=True)\r\n    parser.add_argument(\"--resume\", action=\"store_true\", help=\"Resume only this run directory and identical configuration\")\r\n    args = parser.parse_args()\r\n    if args.model_max_depth is not None and args.model_max_depth <= 0:\r\n        parser.error(\"model-max-depth must be positive\")\r\n\r\n    if not torch.cuda.is_available():\r\n        parser.error(\"CUDA is required for training. CPU preparation is supported; use cpu_smoke.py for a tiny CPU check.\")\r\n    if args.max_depth <= 0 or args.epochs < 1 or args.batch_size < 1 or args.grad_accumulation < 1:\r\n        parser.error(\"Depth, epochs, batch size, and gradient accumulation must be positive\")\r\n    if args.image_size < 56 or args.image_size % 14:\r\n        parser.error(\"image-size must be >=56 and divisible by 14\")\r\n    if args.run_kind == \"trial\" and args.epochs != 1:\r\n        parser.error(\"試験runはepochs=1です。本学習は--run-kind fullと新しいoutput-dirを指定してください。\")\r\n    if args.save_every_epochs < 1 or args.keep_recent_checkpoints < 0:\r\n        parser.error(\"Checkpoint intervals/retention must be positive\")\r\n    if args.workers < 0:\r\n        parser.error(\"workers cannot be negative\")\r\n\r\n    run_dir = args.output_dir.resolve()\r\n    if run_dir.exists() and not args.resume:\r\n        parser.error(f\"Run directory exists. Use --resume only for the same run, or choose a new directory: {run_dir}\")\r\n    if not run_dir.exists() and args.resume:\r\n        parser.error(f\"Cannot resume a run that does not exist: {run_dir}\")\r\n    train_rows = read_manifest(args.train_manifest)\r\n    val_rows = read_manifest(args.val_manifest)\r\n    check_split_isolation(train_rows, val_rows)\r\n    verify_local_pairs(train_rows, args.rgb_root, args.depth_root, \"train\")\r\n    verify_local_pairs(val_rows, args.rgb_root, args.depth_root, \"validation\")\r\n\r\n    seed_all(args.seed)\r\n    device = torch.device(\"cuda\")\r\n    model, init_summary = build_metric_model(\r\n        args.initialization,\r\n        max_depth=args.model_max_depth if args.model_max_depth is not None else args.max_depth,\r\n        relative_model=args.relative_model,\r\n        metric_model=args.metric_model,\r\n        local_files_only=True,\r\n    )\r\n    model_bytes = sum(parameter.numel() * parameter.element_size() for parameter in model.parameters())\r\n    free_before_load, _ = torch.cuda.mem_get_info(device)\r\n    if free_before_load < model_bytes + 256 * 1024**2:\r\n        raise MemoryError(\r\n            f\"GPU safety stop before model transfer: {free_before_load / 1024**3:.2f} GiB free; \"\r\n            f\"model parameters need about {model_bytes / 1024**3:.2f} GiB plus 0.25 GiB margin.\"\r\n        )\r\n    try:\r\n        model.to(device)\r\n    except torch.cuda.OutOfMemoryError as exc:\r\n        torch.cuda.empty_cache()\r\n        raise MemoryError(\"CUDA OOM while placing the model on GPU; no training began. Close other GPU workloads and rerun preflight.\") from exc\r\n    preflight = validate_preflight(args.memory_preflight_report, args, init_summary)\r\n\r\n    train_dataset = UAVDepthDataset(train_rows, args.rgb_root, args.depth_root, image_size=args.image_size, training=True)\r\n    val_dataset = UAVDepthDataset(val_rows, args.rgb_root, args.depth_root, image_size=args.image_size, training=False)\r\n    # Recreated once per epoch with seed+epoch; this makes restarts deterministic at epoch boundaries.\r\n    val_loader = DataLoader(\r\n        val_dataset, batch_size=1, shuffle=False, num_workers=args.workers,\r\n        pin_memory=True, worker_init_fn=worker_seed, persistent_workers=False,\r\n    )\r\n\r\n    backbone_params = [p for name, p in model.named_parameters() if name.startswith(\"backbone.\")]\r\n    head_params = [p for name, p in model.named_parameters() if not name.startswith(\"backbone.\")]\r\n    optimizer = AdamW(\r\n        [{\"params\": backbone_params, \"lr\": args.encoder_lr}, {\"params\": head_params, \"lr\": args.encoder_lr * 10.0}],\r\n        betas=(0.9, 0.999), weight_decay=args.weight_decay,\r\n    )\r\n    optimizer_steps_per_epoch = math.ceil(math.ceil(len(train_dataset) / args.batch_size) / args.grad_accumulation)\r\n    total_steps = optimizer_steps_per_epoch * args.epochs\r\n    scheduler = torch.optim.lr_scheduler.PolynomialLR(optimizer, total_iters=total_steps, power=0.9)\r\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=args.amp)\r\n\r\n    code_files = (\"train.py\", \"modeling.py\", \"data.py\", \"losses.py\", \"metrics.py\", \"audit_data.py\",\r\n                  \"memory_preflight.py\", \"analyze_gt.py\", \"evaluate.py\")\r\n    run_metadata = {\r\n        \"arguments\": json_safe(vars(args)),\r\n        \"initialization\": init_summary,\r\n        \"seed\": args.seed,\r\n        \"versions\": {\r\n            \"python\": sys.version, \"torch\": torch.__version__, \"transformers\": transformers.__version__,\r\n            \"numpy\": np.__version__, \"opencv\": cv2.__version__, \"tifffile\": tifffile.__version__,\r\n            \"huggingface_hub\": huggingface_hub.__version__, \"requests\": requests.__version__,\r\n            \"safetensors\": safetensors.__version__,\r\n        },\r\n        \"device\": torch.cuda.get_device_name(0),\r\n        \"cuda_total_memory_bytes\": torch.cuda.get_device_properties(0).total_memory,\r\n        \"memory_preflight\": preflight,\r\n        \"train_manifest_sha256\": sha256_file(args.train_manifest),\r\n        \"validation_manifest_sha256\": sha256_file(args.val_manifest),\r\n        \"code_sha256\": {filename: sha256_file(Path(__file__).with_name(filename)) for filename in code_files}\r\n        | {\"scene_split_plan.json\": sha256_file(Path(__file__).parent / \"config\" / \"scene_split_plan.json\")},\r\n        \"train_pairs\": len(train_rows), \"validation_pairs\": len(val_rows),\r\n        \"metrics\": [\"mae_m\", \"rmse_m\", \"abs_rel\", \"local_depth_delta_mae_m\", \"pearson_r\"],\r\n        \"loss\": \"SiLog with finite positive targets not exceeding GT max_depth\",\r\n        \"gt_max_depth_m\": args.max_depth,\r\n        \"model_max_depth_m\": init_summary[\"max_depth_m\"],\r\n        \"test_data_read\": False,\r\n    }\r\n    signature_keys = (\r\n        \"initialization\", \"max_depth\", \"model_max_depth\", \"image_size\", \"batch_size\", \"grad_accumulation\", \"epochs\", \"run_kind\",\r\n        \"encoder_lr\", \"weight_decay\", \"workers\", \"seed\", \"amp\", \"save_every_epochs\",\r\n        \"keep_recent_checkpoints\", \"relative_model\", \"metric_model\",\r\n    )\r\n    signature = {key: run_metadata[\"arguments\"].get(key) for key in signature_keys}\r\n    signature.update({\r\n        \"runtime_versions\": {key: run_metadata[\"versions\"][key] for key in (\"torch\", \"transformers\", \"numpy\", \"safetensors\")},\r\n        \"initialization_source\": init_summary[\"source\"],\r\n        \"initialization_source_revision\": init_summary.get(\"source_revision\"),\r\n        \"train_manifest_sha256\": run_metadata[\"train_manifest_sha256\"],\r\n        \"validation_manifest_sha256\": run_metadata[\"validation_manifest_sha256\"],\r\n        \"code_sha256\": run_metadata[\"code_sha256\"],\r\n    })\r\n    metadata_path = run_dir / \"run_metadata.json\"\r\n    if args.resume:\r\n        if not metadata_path.is_file():\r\n            raise FileNotFoundError(f\"Run metadata is missing; refusing unsafe resume: {metadata_path}\")\r\n        old_metadata = json.loads(metadata_path.read_text(encoding=\"utf-8\"))\r\n        old_signature = old_metadata.get(\"resume_signature\")\r\n        validate_resume_signature(old_signature, signature)\r\n    else:\r\n        run_dir.mkdir(parents=True, exist_ok=False)\r\n        run_metadata[\"resume_signature\"] = signature\r\n        write_json_atomic(metadata_path, run_metadata)\r\n\r\n    log_path = run_dir / \"epoch_metrics.jsonl\"\r\n    best_rmse = float(\"inf\")\r\n    best_epoch: int | None = None\r\n    completed_epoch = 0\r\n    epoch_durations: list[float] = []\r\n    if args.resume:\r\n        checkpoints = checkpoint_directories(run_dir)\r\n        if not checkpoints:\r\n            raise FileNotFoundError(f\"No complete checkpoint to resume from in {run_dir}\")\r\n        latest = checkpoints[-1]\r\n        trainer_state = restore_checkpoint(latest, model, optimizer, scheduler, scaler, device)\r\n        completed_epoch = int(trainer_state[\"completed_epoch\"])\r\n        best_rmse_value = trainer_state.get(\"best_validation_rmse_m\")\r\n        best_rmse = float(best_rmse_value) if best_rmse_value is not None else float(\"inf\")\r\n        best_epoch = trainer_state.get(\"best_epoch\")\r\n        if best_epoch is not None:\r\n            best_path = run_dir / f\"checkpoint_epoch_{int(best_epoch):04d}\"\r\n            if not (best_path / \"complete.json\").is_file():\r\n                raise FileNotFoundError(f\"Best checkpoint referenced by training state is missing: {best_path}\")\r\n            write_json_atomic(run_dir / \"best_checkpoint.json\", {\"path\": best_path.name, \"epoch\": int(best_epoch)})\r\n        truncate_epoch_log(log_path, completed_epoch)\r\n        if log_path.is_file():\r\n            for line in log_path.read_text(encoding=\"utf-8\").splitlines():\r\n                try:\r\n                    duration = float(json.loads(line).get(\"epoch_seconds\", 0.0))\r\n                    if duration > 0:\r\n                        epoch_durations.append(duration)\r\n                except (ValueError, json.JSONDecodeError):\r\n                    continue\r\n        print(f\"Resumed from epoch {completed_epoch}/{args.epochs}; remaining {max(args.epochs - completed_epoch, 0)} epoch(s).\")\r\n    else:\r\n        checkpoint = save_checkpoint(run_dir, 0, model, optimizer, scheduler, scaler, best_rmse, best_epoch)\r\n        write_json_atomic(run_dir / \"latest_checkpoint.json\", {\"path\": checkpoint.name, \"completed_epoch\": 0})\r\n        write_json_atomic(run_dir / \"run_status.json\", {\r\n            \"status\": \"running\", \"completed_epochs\": 0, \"remaining_epochs\": args.epochs,\r\n            \"best_validation_rmse_m\": None, \"best_epoch\": None,\r\n        })\r\n\r\n    for epoch_index in range(completed_epoch, args.epochs):\r\n        epoch_number = epoch_index + 1\r\n        print(f\"Starting epoch {epoch_number}/{args.epochs}; completed {epoch_index}, remaining including this epoch {args.epochs - epoch_index}.\")\r\n        seed_all(args.seed + epoch_index)\r\n        train_generator = torch.Generator().manual_seed(args.seed + epoch_index)\r\n        train_loader = DataLoader(\r\n            train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.workers,\r\n            pin_memory=True, drop_last=False, worker_init_fn=worker_seed,\r\n            generator=train_generator, persistent_workers=False,\r\n        )\r\n        epoch_start = time.perf_counter()\r\n        torch.cuda.reset_peak_memory_stats(device)\r\n        model.train()\r\n        running_loss = 0.0\r\n        optimizer_steps_completed = 0\r\n        amp_skipped_steps = 0\r\n        amp_scale_start = scaler.get_scale()\r\n        optimizer.zero_grad(set_to_none=True)\r\n        oom = False\r\n        try:\r\n            number_of_batches = len(train_loader)\r\n            for batch_index, batch in enumerate(train_loader):\r\n                pixels = batch[\"pixel_values\"].to(device, non_blocking=True)\r\n                target = batch[\"depth\"].to(device, non_blocking=True)\r\n                valid = torch.isfinite(target) & (target >= 0.001) & (target <= args.max_depth)\r\n                group_start = (batch_index // args.grad_accumulation) * args.grad_accumulation\r\n                group_size = min(args.grad_accumulation, number_of_batches - group_start)\r\n                with torch.autocast(\"cuda\", dtype=torch.float16, enabled=args.amp):\r\n                    prediction = model(pixel_values=pixels).predicted_depth\r\n                    loss = silog_loss(prediction, target, valid)\r\n                    scaled_loss = loss / group_size\r\n                scaler.scale(scaled_loss).backward()\r\n                running_loss += float(loss.detach().cpu())\r\n                is_update = ((batch_index + 1) % args.grad_accumulation == 0) or (batch_index + 1 == number_of_batches)\r\n                if is_update:\r\n                    scaler.unscale_(optimizer)\r\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)\r\n                    updated = scaled_optimizer_step(optimizer, scheduler, scaler)\r\n                    optimizer_steps_completed += int(updated)\r\n                    amp_skipped_steps += int(not updated)\r\n                    optimizer.zero_grad(set_to_none=True)\r\n                if (batch_index + 1) % 32 == 0:\r\n                    print(f\"TRAIN_BATCH {batch_index+1}/{number_of_batches}; updates={optimizer_steps_completed}; AMP skipped={amp_skipped_steps}; scale={scaler.get_scale()}\", flush=True)\r\n            val_metrics = evaluate_model(model, val_loader, device, args.max_depth)\r\n            torch.cuda.synchronize(device)\r\n        except torch.cuda.OutOfMemoryError as exc:\r\n            oom = True\r\n            optimizer.zero_grad(set_to_none=True)\r\n            torch.cuda.empty_cache()\r\n            write_json_atomic(run_dir / \"run_status.json\", {\r\n                \"status\": \"stopped_cuda_oom\", \"failed_epoch\": epoch_number,\r\n                \"last_completed_epoch\": completed_epoch,\r\n                \"message\": str(exc)[:1000],\r\n                \"resume\": \"Rerun the identical command with --resume after closing other GPU workloads. If resolution or another signed setting changes, start a new run and rerun preflight.\",\r\n            })\r\n            print(f\"CUDA out of memory in epoch {epoch_number}; stopped safely. Latest complete checkpoint: epoch {completed_epoch}. Close other GPU workloads and resume with identical settings; rerun preflight for a new lower-resolution run.\")\r\n        if oom:\r\n            break\r\n\r\n        epoch_seconds = time.perf_counter() - epoch_start\r\n        epoch_durations.append(epoch_seconds)\r\n        mean_epoch_seconds = sum(epoch_durations) / len(epoch_durations)\r\n        remaining_after = args.epochs - epoch_number\r\n        row = {\r\n            \"epoch\": epoch_number,\r\n            \"optimizer_steps_completed\": optimizer_steps_completed,\r\n            \"amp_skipped_steps\": amp_skipped_steps,\r\n            \"amp_scale_start\": amp_scale_start,\r\n            \"amp_scale_end\": scaler.get_scale(),\r\n            \"train_silog\": running_loss / max(len(train_loader), 1),\r\n            \"validation\": val_metrics,\r\n            \"encoder_lr\": optimizer.param_groups[0][\"lr\"],\r\n            \"head_lr\": optimizer.param_groups[1][\"lr\"],\r\n            \"epoch_seconds\": epoch_seconds,\r\n            \"peak_allocated_bytes\": torch.cuda.max_memory_allocated(device),\r\n            \"peak_reserved_bytes\": torch.cuda.max_memory_reserved(device),\r\n            \"free_vram_after_epoch_bytes\": torch.cuda.mem_get_info(device)[0],\r\n            \"mean_epoch_seconds_this_session\": mean_epoch_seconds,\r\n            \"estimated_total_seconds_at_current_rate\": mean_epoch_seconds * args.epochs,\r\n            \"estimated_remaining_seconds\": mean_epoch_seconds * remaining_after,\r\n            \"completed_epochs\": epoch_number,\r\n            \"remaining_epochs\": remaining_after,\r\n        }\r\n        with log_path.open(\"a\", encoding=\"utf-8\") as stream:\r\n            stream.write(json.dumps(row, allow_nan=False) + \"\\n\")\r\n        print(json.dumps(row, allow_nan=False))\r\n\r\n        improved = val_metrics[\"rmse_m\"] < best_rmse\r\n        if improved:\r\n            best_rmse = float(val_metrics[\"rmse_m\"])\r\n            best_epoch = epoch_number\r\n            write_json_atomic(run_dir / \"best_validation_metrics.json\", val_metrics)\r\n        should_save = epoch_number % args.save_every_epochs == 0 or epoch_number == args.epochs or improved\r\n        if should_save:\r\n            checkpoint = save_checkpoint(run_dir, epoch_number, model, optimizer, scheduler, scaler, best_rmse, best_epoch)\r\n            write_json_atomic(run_dir / \"latest_checkpoint.json\", {\"path\": checkpoint.name, \"completed_epoch\": epoch_number})\r\n            if improved:\r\n                write_json_atomic(run_dir / \"best_checkpoint.json\", {\"path\": checkpoint.name, \"epoch\": epoch_number})\r\n            prune_checkpoints(run_dir, best_epoch, args.keep_recent_checkpoints)\r\n            completed_epoch = epoch_number\r\n        write_json_atomic(run_dir / \"run_status.json\", {\r\n            \"status\": \"completed\" if epoch_number == args.epochs else \"running\",\r\n            \"completed_epochs\": completed_epoch,\r\n            \"remaining_epochs\": max(args.epochs - completed_epoch, 0),\r\n            \"best_validation_rmse_m\": best_rmse if math.isfinite(best_rmse) else None,\r\n            \"best_epoch\": best_epoch,\r\n        })\r\n    else:\r\n        write_json_atomic(run_dir / \"run_status.json\", {\r\n            \"status\": \"completed\", \"completed_epochs\": completed_epoch,\r\n            \"remaining_epochs\": max(args.epochs - completed_epoch, 0),\r\n            \"best_validation_rmse_m\": best_rmse if math.isfinite(best_rmse) else None,\r\n            \"best_epoch\": best_epoch,\r\n        })\r\n    print(f\"Run status: {(run_dir / 'run_status.json').read_text(encoding='utf-8')}\")\r\n    if best_epoch is not None:\r\n        print(f\"Best validation RMSE: {best_rmse:.6f} m at epoch {best_epoch}; test data was not read.\")\r\n    if json.loads((run_dir / \"run_status.json\").read_text(encoding=\"utf-8\"))[\"status\"] == \"stopped_cuda_oom\":\r\n        raise SystemExit(2)\r\n\r\n\r\nif __name__ == \"__main__\":\r\n    main()\r\n",'memory_preflight.py':"from __future__ import annotations\r\n\r\nimport argparse\r\nimport json\r\nimport time\r\nfrom pathlib import Path\r\n\r\nimport torch\r\n\r\nif __package__:\r\n    from .losses import silog_loss\r\n    from .modeling import METRIC_OUTDOOR_MODEL_ID, RELATIVE_MODEL_ID, build_metric_model\r\nelse:\r\n    from losses import silog_loss\r\n    from modeling import METRIC_OUTDOOR_MODEL_ID, RELATIVE_MODEL_ID, build_metric_model\r\n\r\n\r\ndef require_finite_gradients(loss, parameters):\r\n    gradients = [p.grad for p in parameters if p.grad is not None]\r\n    if not torch.isfinite(loss).all().item() or not gradients or not all(torch.isfinite(g).all().item() for g in gradients):\r\n        raise RuntimeError(\"損失または勾配に数値異常があります。AMPによる更新スキップをメモリ計測の成功として扱えません。設定を確認して再計測してください。\")\r\n\r\n\r\ndef main() -> None:\r\n    parser = argparse.ArgumentParser(description=\"Measure one synthetic CUDA training step before a real run.\")\r\n    parser.add_argument(\"--initialization\", choices=(\"relative_encoder\", \"metric_outdoor\"), required=True)\r\n    parser.add_argument(\"--relative-model\", default=RELATIVE_MODEL_ID, help=\"Resolved local cache path for Relative Small\")\r\n    parser.add_argument(\"--metric-model\", default=METRIC_OUTDOOR_MODEL_ID, help=\"Resolved local cache path for Metric Outdoor Small\")\r\n    parser.add_argument(\"--image-size\", type=int, default=518)\r\n    parser.add_argument(\"--batch-size\", type=int, default=1)\r\n    parser.add_argument(\"--max-depth\", type=float, required=True)\r\n    parser.add_argument(\"--model-max-depth\", type=float, default=None, help=\"Model output ceiling; --max-depth remains the GT mask limit\")\r\n    parser.add_argument(\"--amp\", action=argparse.BooleanOptionalAction, default=True)\r\n    parser.add_argument(\"--output\", type=Path, required=True)\r\n    args = parser.parse_args()\r\n    if args.model_max_depth is not None and args.model_max_depth <= 0:\r\n        parser.error(\"model-max-depth must be positive\")\r\n    if not torch.cuda.is_available():\r\n        parser.error(\"CUDA is unavailable. This is a GPU-only cell; prior CPU preparation can still be completed.\")\r\n    if args.image_size < 56 or args.image_size % 14 or args.batch_size < 1 or args.max_depth <= 0:\r\n        parser.error(\"image-size must be >=56 and divisible by 14; batch-size and max-depth must be positive\")\r\n    if args.output.exists():\r\n        parser.error(f\"Refusing to overwrite existing preflight report: {args.output}\")\r\n\r\n    device = torch.device(\"cuda\")\r\n    free_before, total_before = torch.cuda.mem_get_info(device)\r\n    result = {\r\n        \"status\": \"running\",\r\n        \"gt_max_depth_m\": args.max_depth,\r\n        \"device\": torch.cuda.get_device_name(0),\r\n        \"total_vram_bytes\": torch.cuda.get_device_properties(0).total_memory,\r\n        \"total_vram_gib\": torch.cuda.get_device_properties(0).total_memory / 1024**3,\r\n        \"free_vram_before_model_bytes\": free_before,\r\n        \"training_feasible\": False,\r\n        \"feasibility_scope\": \"Synthetic square training step with headroom; real data/validation can still OOM.\",\r\n        \"image_size\": args.image_size,\r\n        \"batch_size\": args.batch_size,\r\n        \"amp\": args.amp,\r\n        \"initialization\": None,\r\n        \"peak_allocated_bytes_one_synthetic_step\": None,\r\n        \"peak_reserved_bytes_one_synthetic_step\": None,\r\n        \"optimizer_step\": \"synthetic-only; no dataset samples or checkpoint were used\",\r\n    }\r\n    started = time.perf_counter()\r\n    try:\r\n        torch.manual_seed(42)\r\n        torch.cuda.manual_seed_all(42)\r\n        model, init_summary = build_metric_model(\r\n            args.initialization,\r\n            max_depth=args.model_max_depth if args.model_max_depth is not None else args.max_depth,\r\n            relative_model=args.relative_model,\r\n            metric_model=args.metric_model,\r\n            local_files_only=True,\r\n        )\r\n        result[\"initialization\"] = init_summary\r\n        model = model.to(device).train()\r\n        total_params = sum(parameter.numel() for parameter in model.parameters())\r\n        trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\r\n        result[\"parameters\"] = total_params\r\n        result[\"trainable_parameters\"] = trainable_params\r\n        result[\"static_parameter_gradient_adam_bytes_lower_bound\"] = 4 * total_params + 12 * trainable_params\r\n        result[\"static_lower_bound_gib_excludes_activations\"] = (4 * total_params + 12 * trainable_params) / 1024**3\r\n        torch.cuda.synchronize(device)\r\n        baseline_allocated = torch.cuda.memory_allocated(device)\r\n        torch.cuda.empty_cache()\r\n        baseline_reserved = torch.cuda.memory_reserved(device)\r\n        torch.cuda.reset_peak_memory_stats(device)\r\n        result[\"baseline_model_allocated_bytes\"] = baseline_allocated\r\n        result[\"baseline_model_reserved_bytes\"] = baseline_reserved\r\n\r\n        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)\r\n        pixels = torch.randn(args.batch_size, 3, args.image_size, args.image_size, device=device)\r\n        target = (torch.rand(args.batch_size, args.image_size, args.image_size, device=device) * args.max_depth).clamp_min(0.1)\r\n        valid = torch.isfinite(target) & (target > 0)\r\n        scaler = torch.amp.GradScaler(\"cuda\", enabled=args.amp)\r\n        attempts = []\r\n        for attempt in range(9):\r\n            optimizer.zero_grad(set_to_none=True)\r\n            with torch.autocast(\"cuda\", dtype=torch.float16, enabled=args.amp):\r\n                prediction = model(pixel_values=pixels).predicted_depth\r\n                loss = silog_loss(prediction, target, valid)\r\n            if not torch.isfinite(loss).all().item():\r\n                raise RuntimeError(\"Nonfinite forward loss; preflight stopped\")\r\n            scaler.scale(loss).backward()\r\n            scaler.unscale_(optimizer)\r\n            try:\r\n                require_finite_gradients(loss, model.parameters())\r\n                attempts.append({\"scale\":scaler.get_scale(), \"finite_gradients\":True})\r\n                break\r\n            except RuntimeError:\r\n                attempts.append({\"scale\":scaler.get_scale(), \"finite_gradients\":False})\r\n                if not args.amp or attempt==8:\r\n                    raise\r\n                # Real AdamW step is still mandatory after adaptation; overflow\r\n                # attempts are never counted as successful optimizer steps.\r\n                scaler.update(new_scale=scaler.get_scale()/2)\r\n        result[\"amp_scale_attempts\"] = attempts\r\n        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)\r\n        scaler.step(optimizer)\r\n        scaler.update()\r\n        if not optimizer.state:\r\n            raise RuntimeError(\"optimizer状態が作成されませんでした。事前計測を成功と判定しません。\")\r\n        torch.cuda.synchronize(device)\r\n        result.update({\r\n            \"status\": \"pass\",\r\n            \"peak_allocated_bytes_one_synthetic_step\": torch.cuda.max_memory_allocated(device),\r\n            \"peak_reserved_bytes_one_synthetic_step\": torch.cuda.max_memory_reserved(device),\r\n            \"loss\": float(loss.detach().cpu()),\r\n            \"elapsed_seconds\": time.perf_counter() - started,\r\n        })\r\n        required = baseline_reserved + int(max(result[\"peak_reserved_bytes_one_synthetic_step\"] - baseline_reserved, 0) * 1.15) + 512 * 1024**2\r\n        result[\"required_vram_with_margin_bytes\"] = required\r\n        result[\"free_vram_after_step_bytes\"] = torch.cuda.mem_get_info(device)[0]\r\n        result[\"training_feasible\"] = free_before >= required\r\n        if not result[\"training_feasible\"]:\r\n            result[\"status\"] = \"insufficient_headroom\"\r\n            result[\"message\"] = \"合成stepは完了しましたがVRAMの安全余裕が不足しています。学習を開始せず設定を確認してください。\"\r\n    except torch.cuda.OutOfMemoryError as exc:\r\n        result[\"peak_allocated_bytes_one_synthetic_step\"] = torch.cuda.max_memory_allocated(device)\r\n        result[\"peak_reserved_bytes_one_synthetic_step\"] = torch.cuda.max_memory_reserved(device)\r\n        torch.cuda.empty_cache()\r\n        result.update({\"status\": \"stopped_cuda_oom\", \"message\": str(exc)[:1000], \"elapsed_seconds\": time.perf_counter() - started})\r\n    except Exception as exc:\r\n        result.update({\"status\": \"failed\", \"message\": f\"{type(exc).__name__}: {exc}\", \"elapsed_seconds\": time.perf_counter() - started})\r\n        args.output.parent.mkdir(parents=True, exist_ok=True)\r\n        args.output.write_text(json.dumps(result, indent=2, allow_nan=False), encoding=\"utf-8\")\r\n        raise\r\n    args.output.parent.mkdir(parents=True, exist_ok=True)\r\n    args.output.write_text(json.dumps(result, indent=2, allow_nan=False), encoding=\"utf-8\")\r\n    print(json.dumps(result, indent=2, allow_nan=False))\r\n    if result[\"status\"] != \"pass\":\r\n        raise SystemExit(\"GPU事前計測が通りませんでした。学習を開始せずレポートを確認してください。392px/280pxは手動変更後に再計測してください。\")\r\n\r\n\r\nif __name__ == \"__main__\":\r\n    main()\r\n",'continue_gpu.py':"from run_experiment2 import *\ndef main():\n    ap=argparse.ArgumentParser(); ap.add_argument('--project',type=Path,required=True); ap.add_argument('--work',type=Path,required=True); ap.add_argument('--run-trial',action='store_true'); args=ap.parse_args()\n    assert args.run_trial and torch.cuda.is_available()\n    project=args.project; work=args.work; source=Path(__file__).parent\n    plan=project/'uav_metric_finetuning/config/verified_plan'; lr=Path('/content/uav3dcrop/rgb'); ld=Path('/content/uav3dcrop/depth')\n    assert json.loads((work/'ssd_integrity.json').read_text())['status']=='pass'\n    assert dataset_status(plan,lr,ld,['train','validation'])['complete']\n    assert json.loads((work/'output_cap_probe.json').read_text())['actual_median_ratio']==0.8125\n    run=project/'runs/metric_outdoor_trial2_20261013_s42_518_cap80_gt65'; assert not run.exists(), 'Existing run preserved'\n    metric=project/'model_cache/models--depth-anything--Depth-Anything-V2-Metric-Outdoor-Small-hf/snapshots/fd2c22027eaf20374204f14099b8341e1925ad39'\n    common=['--initialization','metric_outdoor','--metric-model',str(metric),'--max-depth','65','--model-max-depth','80','--image-size','518','--batch-size','1','--amp']\n    preflight=work/'metric_outdoor_preflight_retry.json'\n    common=['--initialization','metric_outdoor','--metric-model',str(metric),'--max-depth','65','--model-max-depth','80','--image-size','518','--batch-size','1','--amp']\n    subprocess.run([sys.executable,'-u',str(source/'memory_preflight.py'),*common,'--output',str(preflight)],check=True)\n    assert json.loads(preflight.read_text())['status']=='pass'\n    train_start=time.monotonic()\n    subprocess.run([sys.executable,'-u',str(source/'train.py'),*common,'--train-manifest',str(plan/'train.csv'),'--val-manifest',str(plan/'validation.csv'),'--rgb-root',str(lr),'--depth-root',str(ld),'--epochs','1','--run-kind','trial','--grad-accumulation','4','--seed','42','--workers','2','--encoder-lr','5e-6','--save-every-epochs','1','--keep-recent-checkpoints','0','--memory-preflight-report',str(preflight),'--output-dir',str(run)],check=True)\n    assert json.loads((run/'run_status.json').read_text())['status']=='completed'\n    checkpoints={}\n    for cp in run.glob('checkpoint_epoch_*'):\n        marker=json.loads((cp/'complete.json').read_text())\n        assert all(sha(cp/n)==h for n,h in marker['file_sha256'].items())\n        checkpoints[cp.name]=marker\n    newjson(work/'trial_complete.json',{'run':str(run),'training_total_seconds_including_startup_save':time.monotonic()-train_start,'checkpoints_verified':checkpoints,'model_output_ceiling_m':80,'gt_mask_max_m':65,'epochs':1,'test_used':False})\n    spec={'manifest':str(plan/'validation.csv'),'validation_sha256':sha(plan/'validation.csv'),'rgb_root':str(lr),'depth_root':str(ld),'output':str(work/'analysis'),'models':{'A':str(project/'runs/relative_encoder_trial_20261013_s42_518/checkpoint_epoch_0001'),'B':str(metric),'C':str(run/'checkpoint_epoch_0001')},'model_caps':{'A':65,'B':80,'C':80}}\n    newjson(work/'analysis_spec.json',spec)\n    subprocess.run([sys.executable,'-u',str(source/'shape_analysis.py'),'--spec',str(work/'analysis_spec.json'),'--phase','infer'],check=True)\n    print('ALL_GPU_WORK_COMPLETE. Persisted training and prediction cache. Switch to CPU for analysis.',flush=True)\n\nif __name__=='__main__':main()\n"}
for name,source_text in changes.items():
    ast.parse(source_text)
    p=NEW_EXPERIMENT_CODE/name
    p.write_text(source_text,encoding='utf-8')
print('AMP事前計測のみ再試行。コピー・上限検査・監査は再実行しません。')
EXPERIMENT_CODE=NEW_EXPERIMENT_CODE
experiment_log=EXPERIMENT_DIR/'gpu_execution_retry.log'
with experiment_log.open('x') as f:
    experiment_process=subprocess.Popen([sys.executable,'-u',str(EXPERIMENT_CODE/'continue_gpu.py'),'--project',str(PROJECT_DIR),'--work',str(EXPERIMENT_DIR),'--run-trial'],stdout=f,stderr=subprocess.STDOUT,env={**os.environ,'USE_TF':'0','PYTHONUNBUFFERED':'1'})
print('継続PID:',experiment_process.pid,'修正版:',EXPERIMENT_CODE,flush=True)


In [ ]:
import hashlib
original_stamp='20261013T090441609126Z'
old_audit_path=WORK_DIR/f'data_audit_{original_stamp}.json'
old_gt_path=WORK_DIR/f'gt_distribution_train_{original_stamp}.json'
old_audit=json.loads(old_audit_path.read_text())
old_gt=json.loads(old_gt_path.read_text())
current_audit_context=preparation_context(**audit_context_args())
current_gt_context=preparation_context(**gt_context_args())
for kind,old,current in [('audit',old_audit,current_audit_context),('gt',old_gt,current_gt_context)]:
    old_context=old['reuse_context']
    print(kind,'changed keys:',[k for k in current if current[k]!=old_context.get(k)])
    print(kind,'old packages:',old_context['packages'],'new packages:',current['packages'])
    changed=[(k,v,current['files'].get(k)) for k,v in old_context['files'].items() if current['files'].get(k)!=v]
    print(kind,'changed file count:',len(changed),'examples:',changed[:3])
print('GPU:',torch.cuda.get_device_name(0),'VRAM total/free GiB:',torch.cuda.get_device_properties(0).total_memory/1024**3,torch.cuda.mem_get_info()[0]/1024**3)
print('データ取得状態:',dataset_status(PLAN_DIR,DRIVE_RGB_ROOT,DRIVE_DEPTH_ROOT,['train','validation']))


In [ ]:
RUN_CONTENT_REVALIDATION = True
if RUN_CONTENT_REVALIDATION:
    import csv,copy,time
    from concurrent.futures import ThreadPoolExecutor,as_completed
    from runtime_state import sha256,safe_file
    for old,current in [(old_audit,current_audit_context),(old_gt,current_gt_context)]:
        assert [k for k in current if current[k]!=old['reuse_context'].get(k)]==['files'], '時刻以外の条件変更: 再監査が必要'
        assert set(current['files'])==set(old['reuse_context']['files'])
        for key,value in current['files'].items():
            previous=old['reuse_context']['files'][key]
            assert value['bytes']==previous['bytes'], 'サイズ変更: 停止'
            assert abs(value['mtime_ns']-previous['mtime_ns'])<=1_000_000_000, '1秒を超える時刻変更: 停止'
    records={}
    for split in ['train','validation']:
        with (PLAN_DIR/f'{split}.csv').open(encoding='utf-8-sig',newline='') as f:
            for row in csv.DictReader(f):
                for root,key,sizekey,hashkey in [(DRIVE_RGB_ROOT,'image_relpath','image_size_bytes','image_lfs_oid'),(DRIVE_DEPTH_ROOT,'depth_relpath','depth_size_bytes','depth_lfs_oid')]:
                    records[f'{split}:{key}:{row[key]}']=(safe_file(root,row[key]),int(row[sizekey]),row[hashkey])
    changed_keys=[k for k,v in current_audit_context['files'].items() if v!=old_audit['reuse_context']['files'][k]]
    print('変更時刻を検出したファイルのみSHA256検証:',len(changed_keys),'件。元ファイル・旧レポートは保持。',flush=True)
    def verify_changed(key):
        path,size,expected=records[key]
        assert len(expected)==64 and path.stat().st_size==size
        actual=sha256(path)
        if actual!=expected: raise RuntimeError(f'SHA256不一致。学習停止: {path}')
        return key,{'sha256':actual,'bytes':size}
    verified_hashes={}
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures=[pool.submit(verify_changed,k) for k in changed_keys]
        for number,future in enumerate(as_completed(futures),1):
            key,value=future.result(); verified_hashes[key]=value
            if number%10==0 or number==len(changed_keys): print('SHA256検証完了:',number,'/',len(changed_keys),flush=True)
    gt_changed={k for k,v in current_gt_context['files'].items() if v!=old_gt['reuse_context']['files'][k]}
    assert gt_changed<=set(verified_hashes)
    assert preparation_context(**audit_context_args())==current_audit_context
    assert preparation_context(**gt_context_args())==current_gt_context
    migration_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    for kind,old,old_path,current,prefix in [('audit',old_audit,old_audit_path,current_audit_context,'data_audit'),('gt',old_gt,old_gt_path,current_gt_context,'gt_distribution_train')]:
        derived=copy.deepcopy(old)
        derived['reuse_context']=current
        derived['content_revalidation']={'method':'unchanged specs/code/manifests/roots/sizes; <=1sec Drive timestamp rounding; changed files exactly match pinned LFS SHA256','source_report':str(old_path),'source_report_sha256':sha256(old_path),'verified_changed_files':verified_hashes,'pixel_audit_reexecuted':False}
        target=WORK_DIR/f'{prefix}_content_verified_{migration_stamp}.json'
        with target.open('x') as f: json.dump(derived,f,indent=2)
        print('内容一致で引き継いだレポート:',target,flush=True)
    refresh_preparation_state()
    assert AUDIT_OK and GT_REPORT is not None
    print('監査・GT再利用成功。ファイル内容を確認し、重い画素再走査を回避しました。')
else: print('RUN_CONTENT_REVALIDATION=False: レポート移行を実行しません。')


In [ ]:
import re
cpu_hash_records={}
cpu_hash_log_paths=[WORK_DIR/f'download_{original_stamp}.log',WORK_DIR/f'download_parallel_{original_stamp}.log']
for log_path in cpu_hash_log_paths:
    for line in log_path.read_text().splitlines():
        match=re.match(r'^\[\d+/\d+\] (\S+) (\S+) \(([^,]+), ([^)]+)\)$',line)
        if not match: continue
        scene,filename,image_hash,depth_hash=match.groups()
        for key,value in [(f'{scene}/images/{filename}',image_hash),(f'{scene}_depth/{Path(filename).stem}_depth.tif',depth_hash)]:
            if re.fullmatch('[0-9a-f]{64}',value):
                assert key not in cpu_hash_records or cpu_hash_records[key]==value
                cpu_hash_records[key]=value
print('CPUログの内容SHA256:',len(cpu_hash_records),'ファイル')
missing_cpu_anchors=[]
for key,(path,size,expected) in list(records.items()):
    relative=key.split(':',2)[2]
    anchor=cpu_hash_records.get(relative)
    if anchor: records[key]=(path,size,anchor)
    elif key in changed_keys: missing_cpu_anchors.append(key)
print('77件のうちCPUハッシュ記録がないファイル:',missing_cpu_anchors)


In [ ]:
RUN_CONTENT_REVALIDATION = True
if RUN_CONTENT_REVALIDATION:
    assert not missing_cpu_anchors, 'CPU時点の内容を証明できないファイルがあります。再監査が必要'
    verified_hashes={}
    verification_start=time.monotonic()
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures=[pool.submit(verify_changed,k) for k in changed_keys]
        for number,future in enumerate(as_completed(futures),1):
            key,value=future.result(); verified_hashes[key]=value
            if number%10==0 or number==len(changed_keys): print('CPU SHA256との一致:',number,'/',len(changed_keys),flush=True)
    gt_changed={k for k,v in current_gt_context['files'].items() if v!=old_gt['reuse_context']['files'][k]}
    assert gt_changed<=set(verified_hashes)
    assert preparation_context(**audit_context_args())==current_audit_context
    assert preparation_context(**gt_context_args())==current_gt_context
    migration_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    source_log_hashes={str(p):sha256(p) for p in cpu_hash_log_paths}
    for kind,old,old_path,current,prefix in [('audit',old_audit,old_audit_path,current_audit_context,'data_audit'),('gt',old_gt,old_gt_path,current_gt_context,'gt_distribution_train')]:
        derived=copy.deepcopy(old)
        derived['reuse_context']=current
        derived['content_revalidation']={'method':'identical specs/code/packages/manifests/roots/sizes; <=1sec Drive timestamp rounding; changed files content-identical to CPU download SHA256 logs','source_report':str(old_path),'source_report_sha256':sha256(old_path),'source_log_sha256':source_log_hashes,'verified_changed_files':verified_hashes,'pixel_audit_reexecuted':False,'verification_seconds':time.monotonic()-verification_start}
        target=WORK_DIR/f'{prefix}_content_verified_{migration_stamp}.json'
        with target.open('x') as f: json.dump(derived,f,indent=2)
        print('内容一致で引き継いだレポート:',target,flush=True)
    refresh_preparation_state()
    assert AUDIT_OK and GT_REPORT is not None
    print('77件の内容SHA256一致。監査・GT再利用成功。重い画素再走査は不要。')
else: print('RUN_CONTENT_REVALIDATION=False: 検証を実行しません。')


## 2. CPU: 分割計画と転送量

通常は同梱したrevision固定の計画を再利用します。再照会は小さなtransforms.jsonとファイルメタデータのみで、RGB・深度本体は取得しません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_METADATA_PLAN = False
if RUN_METADATA_PLAN:
    if GENERATED_PLAN_DIR.exists() and any(GENERATED_PLAN_DIR.iterdir()):
        raise FileExistsError(f'上書き防止。別フォルダーを使ってください: {GENERATED_PLAN_DIR}')
    GENERATED_PLAN_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, str(CODE_DIR/'build_remote_plan.py'), '--output-dir', str(GENERATED_PLAN_DIR)], check=True)
    PLAN_DIR = GENERATED_PLAN_DIR
plan_summary = json.loads((PLAN_DIR/'plan_summary.json').read_text(encoding='utf-8'))
print('RGB/depth dataset commits:', plan_summary['rgb_revision'], plan_summary['depth_revision'])
for split, count in plan_summary['selected_pair_counts'].items():
    info = plan_summary['selected_bytes_by_split'][split]
    print(f'{split}: {count} pairs, {info["bytes"]:,} bytes ({info["gib"]:.3f} GiB)')
for split, dist in plan_summary['split_distributions'].items():
    print(split, '| crops:', dist['crops'], '| views:', dist['views'], '| days:', dist['days'])
print('Metadata missing candidates:')
for scene in plan_summary.get('scene_metadata', []):
    for item in scene.get('missing_candidate_details', []):
        print(scene['split'], scene['scene'], item)
train_val_bytes = sum(int(plan_summary['selected_bytes_by_split'][s]['bytes']) for s in ('train','validation'))
train_val_gib = train_val_bytes / 1024**3
print(f'train+validation: {train_val_bytes:,} bytes = {train_val_gib:.2f} GiB')
print(f'Drive + local copy needs about {2*train_val_gib:.2f} GiB raw; target at least {2*train_val_gib*1.1:.2f} GiB free.')

## 3. CPU: データ取得ゲート（初期値OFF）

取得先はDriveです。必要量・空き容量を確認後、実行する場合はRUN_DOWNLOAD=Trueに加え、確認欄へ計画の正確なbyte数を入力します。同サイズの既存ファイルは再利用します。independent_testは最終モデル確定後まで取得しません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_DOWNLOAD = False
SPLITS_TO_DOWNLOAD = ['train', 'validation']  # independent_testは最終評価まで無効
CONFIRM_PLAN_TOTAL_BYTES = None  # train+validation: 7850330776
status = dataset_status(PLAN_DIR, DRIVE_RGB_ROOT, DRIVE_DEPTH_ROOT, SPLITS_TO_DOWNLOAD)
print('取得状況:', status)
if status['conflicting_files']:
    raise RuntimeError('サイズが異なる既存ファイルがあります。上書きせず、保存先を確認してください。')
if RUN_DOWNLOAD:
    if CONFIRM_PLAN_TOTAL_BYTES != status['planned_bytes']:
        raise RuntimeError('転送しません。計画とDriveの空きを確認後、CONFIRM_PLAN_TOTAL_BYTESを入力してください。')
    subprocess.run([sys.executable,str(CODE_DIR/'fetch_selected_data.py'),'--plan-dir',str(PLAN_DIR),
        '--rgb-root',str(DRIVE_RGB_ROOT),'--depth-root',str(DRIVE_DEPTH_ROOT),
        '--splits',*SPLITS_TO_DOWNLOAD,'--download','--confirm-total-bytes',str(status['planned_bytes'])],check=True)
    status = dataset_status(PLAN_DIR, DRIVE_RGB_ROOT, DRIVE_DEPTH_ROOT, SPLITS_TO_DOWNLOAD)
if status['complete']:
    print(f"取得完了: {status['complete_pairs']}組、{status['present_files']}ファイルの存在とサイズを確認しました。再ダウンロードは不要です。")
else:
    print(f"未取得・不完全: {status['missing_files']}ファイル。実行する場合だけRUN_DOWNLOAD=Trueにしてください。")


## 4. CPU: DriveからローカルSSDへ任意コピー

Drive直読み込みが遅いときだけ/contentへコピーします。必要容量を計算し、空きが足りなければ停止します。Driveの元ファイルは保持されます。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_STAGE_TO_LOCAL = False
STAGE_SPLITS = ['train', 'validation']
if RUN_STAGE_TO_LOCAL:
    subprocess.run([sys.executable,str(CODE_DIR/'stage_selected_data.py'),'--plan-dir',str(PLAN_DIR),
        '--source-rgb-root',str(DRIVE_RGB_ROOT),'--source-depth-root',str(DRIVE_DEPTH_ROOT),
        '--target-rgb-root',str(LOCAL_RGB_ROOT),'--target-depth-root',str(LOCAL_DEPTH_ROOT),
        '--splits',*STAGE_SPLITS,'--stage'],check=True)
    ACTIVE_RGB_ROOT, ACTIVE_DEPTH_ROOT = LOCAL_RGB_ROOT, LOCAL_DEPTH_ROOT
else: ACTIVE_RGB_ROOT, ACTIVE_DEPTH_ROOT = DRIVE_RGB_ROOT, DRIVE_DEPTH_ROOT
print('Active dataset roots:', ACTIVE_RGB_ROOT, ACTIVE_DEPTH_ROOT)

## 5. CPU: 対応・画素・漏洩監査

3つのmanifestすべてでscene/day分割漏洩を確認します。物理ファイルとTIFF画素は既定でtrain/validationだけを監査します。未取得ファイルがあれば不足数を報告し、画素監査をスキップします。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_DATA_AUDIT = False
FORCE_REAUDIT = False
refresh_preparation_state()
if AUDIT_OK and not FORCE_REAUDIT:
    print('正常な監査結果を再利用します:', AUDIT_PATH)
elif RUN_DATA_AUDIT:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    audit_path = WORK_DIR/f'data_audit_{stamp}.json'
    subprocess.run([sys.executable,str(CODE_DIR/'audit_data.py'),
        '--train',str(PLAN_DIR/'train.csv'),'--validation',str(PLAN_DIR/'validation.csv'),
        '--independent-test',str(PLAN_DIR/'independent_test.csv'),
        '--physical-splits','train','validation','--plan-summary',str(PLAN_DIR/'plan_summary.json'),
        '--rgb-root',str(DRIVE_RGB_ROOT),'--depth-root',str(DRIVE_DEPTH_ROOT),'--output',str(audit_path)],check=True)
    refresh_preparation_state()
    print('監査結果:', audit_path, '| pass:', AUDIT_OK)
else:
    print('データ取得後、RUN_DATA_AUDIT=Trueで監査してください。未取得なら不足数を表示してスキップします。')


## 6. CPU: train-only GT深度分布

GTを読み込むため、データ取得と監査の後に実行します。解析対象はtrain manifestだけです。max_depthの候補を出しますが、分布を確認してから設定してください。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_GT_ANALYSIS = False
FORCE_GT_ANALYSIS = False
refresh_preparation_state()
if GT_REPORT and not FORCE_GT_ANALYSIS:
    print('正常なtrain-only GT解析を再利用:', GT_REPORT_PATH)
elif RUN_GT_ANALYSIS:
    if not AUDIT_OK:
        raise RuntimeError('データ取得と監査passが必要です。先に監査セルを実行してください。')
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output = WORK_DIR/f'gt_distribution_train_{stamp}.json'
    subprocess.run([sys.executable,str(CODE_DIR/'analyze_gt.py'),'--manifest',str(PLAN_DIR/'train.csv'),
        '--depth-root',str(DRIVE_DEPTH_ROOT),'--seed',str(GT_SEED),
        '--max-pixels-per-frame',str(GT_MAX_PIXELS),'--output',str(output)],check=True)
    refresh_preparation_state()
if GT_REPORT:
    print('train GT分位点 (m):', GT_REPORT['quantiles_m'])
    print('max_depth候補 (m):', GT_REPORT['suggested_max_depth_m'])
    print('範囲・単位を確認後、設定セルのOVERRIDESでMAX_DEPTH_Mを指定し保存してください。')
else:
    print('RUN_GT_ANALYSIS=Trueでtrainだけを解析してください。max_depthはまだ設定しません。')


## 7. CPU: 学習条件

既定は1方式・1epoch・518解像度です。比較方式を試すときは1方式ずつ切り替えます。両方の連続実行はTRAINING_MODESへ2方式を明記した場合だけです。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

# 初期値: relative_encoder / trial / EPOCHS=1 / 518 / batch1 / accum4 / AMP / seed42
# 初回GT確認後の例: OVERRIDES={'MAX_DEPTH_M': 確認した値}
# 本学習の例: {'RUN_KIND':'full','EPOCHS':20,'RUN_LABEL':'full_s42_v1','RESUME_TRAINING':False}
# 同じ本学習の中断再開: {'RESUME_TRAINING':True}（他の設定はそのまま）
OVERRIDES = {}
SAVE_SETTINGS = False  # 編集をDriveに保存するときだけTrue
TRAIN_CONFIG = validate_settings(TRAIN_CONFIG | OVERRIDES)
globals().update(TRAIN_CONFIG)
if SAVE_SETTINGS:
    save_settings(SETTINGS_PATH, TRAIN_CONFIG)
    print('設定保存完了:', SETTINGS_PATH)
else:
    print('表示中の設定はDriveから復元/このセッションのみ。変更時はSAVE_SETTINGS=Trueで保存してください。')
print(json.dumps(TRAIN_CONFIG,ensure_ascii=False,indent=2))


## 8. CPU: モデルキャッシュ確認・取得ゲート

固定revisionをDrive上のHub cacheへ保存し、再実行時は再利用します。ALLOW_MODEL_DOWNLOAD=Falseのときはローカルキャッシュだけを探し、ネットワークアクセスしません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

from huggingface_hub import snapshot_download
from modeling import RELATIVE_MODEL_ID,METRIC_OUTDOOR_MODEL_ID,RELATIVE_MODEL_REVISION,METRIC_OUTDOOR_MODEL_REVISION
ALLOW_MODEL_DOWNLOAD = False  # 重み取得を許可する場合のみTrue
MODEL_PATHS = {}
specs = {
 'relative':(RELATIVE_MODEL_ID,RELATIVE_MODEL_REVISION),
 'metric':(METRIC_OUTDOOR_MODEL_ID,METRIC_OUTDOOR_MODEL_REVISION)
}
needed = sorted({'relative' if m=='relative_encoder' else 'metric' for m in TRAINING_MODES})
for key in needed:
    repo,revision = specs[key]
    try:
        path = snapshot_download(repo_id=repo,revision=revision,cache_dir=str(MODEL_CACHE_DIR),
            local_files_only=not ALLOW_MODEL_DOWNLOAD,allow_patterns=['config.json','*.safetensors'])
        MODEL_PATHS[key] = str(path)
        print('モデルcache確認成功:',key,path)
    except Exception as exc:
        print('モデル未取得/取得失敗:',repo,revision,'|',type(exc).__name__,str(exc))
        print('取得許可:',ALLOW_MODEL_DOWNLOAD,'| cacheに無ければ取得を許可するときだけALLOW_MODEL_DOWNLOAD=Trueにして、このセルのみ再実行してください。通信失敗時はモデルrevision、Drive空き容量、ネットワークを確認してください。')

In [ ]:
if not globals().get('SESSION_READY',False): raise RuntimeError('最初のDrive接続・状態復元セルを実行してください。')
RUN_GPU_PREFLIGHT = True
prepared_notebook=json.loads((CODE_DIR/'colab_train.ipynb').read_text())
preflight_source=prepared_notebook['cells'][19]['source']
preflight_source=''.join(preflight_source) if isinstance(preflight_source,list) else preflight_source
assert preflight_source.count('RUN_GPU_PREFLIGHT = False')==1
exec(preflight_source.replace('RUN_GPU_PREFLIGHT = False','RUN_GPU_PREFLIGHT = True'),globals())
print('GPU事前計測レポート:',PREFLIGHT_REPORTS)
for mode,path in PREFLIGHT_REPORTS.items(): print(mode,json.dumps(json.loads(path.read_text()),ensure_ascii=False,indent=2))


In [ ]:
import os,time
RUN_STAGE_TO_LOCAL = True
assert AUDIT_OK and GT_REPORT is not None
assert json.loads(PREFLIGHT_REPORTS['relative_encoder'].read_text())['status']=='pass'
gpu_run_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
stage_log=WORK_DIR/f'stage_local_{gpu_run_stamp}.log'
stage_start=time.monotonic()
if RUN_STAGE_TO_LOCAL:
    with stage_log.open('x') as log:
        stage_process=subprocess.Popen([sys.executable,'-u',str(CODE_DIR/'stage_selected_data.py'),'--plan-dir',str(PLAN_DIR),'--source-rgb-root',str(DRIVE_RGB_ROOT),'--source-depth-root',str(DRIVE_DEPTH_ROOT),'--target-rgb-root',str(LOCAL_RGB_ROOT),'--target-depth-root',str(LOCAL_DEPTH_ROOT),'--splits','train','validation','--stage'],stdout=log,stderr=subprocess.STDOUT)
    print('ローカルSSDへのコピーを開始。Drive原本を保持。PID:',stage_process.pid,'ログ:',stage_log,flush=True)
else: print('ローカルコピーOFF')


In [ ]:
if not globals().get('SESSION_READY',False): raise RuntimeError('最初のDrive接続・状態復元セルを実行してください。')
assert RUN_KIND=='trial' and EPOCHS==1 and TRAINING_MODES==['relative_encoder']
assert IMAGE_SIZE==518 and BATCH_SIZE==1 and GRAD_ACCUMULATION==4 and USE_AMP and SEED==42 and MAX_DEPTH_M==65.0
print('ローカルコピーの完了を待ちます。',flush=True)
stage_result=stage_process.wait()
print('コピー結果:',stage_result,'所要秒:',time.monotonic()-stage_start,flush=True)
print(stage_log.read_text()[-1500:],flush=True)
assert stage_result==0, 'ローカルコピー失敗。学習を開始しません。'
local_status=dataset_status(PLAN_DIR,LOCAL_RGB_ROOT,LOCAL_DEPTH_ROOT,['train','validation'])
assert local_status['complete'] and local_status['complete_pairs']==512 and local_status['present_files']==1024
ACTIVE_RGB_ROOT,ACTIVE_DEPTH_ROOT=LOCAL_RGB_ROOT,LOCAL_DEPTH_ROOT
RUN_TRAINING = True
training_log=WORK_DIR/f'trial_training_{gpu_run_stamp}.log'
trial_start=time.monotonic()
train_source=prepared_notebook['cells'][21]['source']
train_source=''.join(train_source) if isinstance(train_source,list) else train_source
assert train_source.count('RUN_TRAINING = False')==1 and train_source.count('subprocess.run(cmd, check=True)')==1
train_source=train_source.replace('RUN_TRAINING = False','RUN_TRAINING = True')
background_call='train_process = subprocess.Popen(cmd, stdout=training_log_stream, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONUNBUFFERED":"1"}); print("1エポック試験学習を開始。PID:",train_process.pid,"ログ:",training_log,flush=True)'
train_source=train_source.replace('subprocess.run(cmd, check=True)',background_call)
with training_log.open('x') as training_log_stream: exec(train_source,globals())
trial_run_dir=RUNS_DIR/f'relative_encoder_{RUN_LABEL}'
with (WORK_DIR/f'trial_started_{gpu_run_stamp}.json').open('x') as f: json.dump({'config':TRAIN_CONFIG,'run_dir':str(trial_run_dir),'log':str(training_log),'pid':train_process.pid,'local_data':local_status,'preflight':str(PREFLIGHT_REPORTS['relative_encoder']),'started_utc':datetime.now(timezone.utc).isoformat(),'compute_policy':'free T4 only; no CU purchase'},f,indent=2)
print('チェックポイント保存先:',trial_run_dir,flush=True)


In [ ]:
import psutil, hashlib, csv
from pathlib import Path
print('既存プロセス:', [(p.pid,p.info['cmdline']) for p in psutil.process_iter(['cmdline']) if any('stage_selected_data.py' in s or '/train.py' in s for s in (p.info['cmdline'] or []))])
print('コピーprocess:', globals().get('stage_process').poll() if globals().get('stage_process') else 'handleなし')
print('学習process:', globals().get('train_process').poll() if globals().get('train_process') else '未起動')
print('SSD整合性:',dataset_status(PLAN_DIR,LOCAL_RGB_ROOT,LOCAL_DEPTH_ROOT,['train','validation']))
print('Drive整合性:',dataset_status(PLAN_DIR,DRIVE_RGB_ROOT,DRIVE_DEPTH_ROOT,['train','validation']))
print('残ったコピー一時ファイル:', list(Path('/content/uav3dcrop').rglob('*.copying')))
trial_run_dir=RUNS_DIR/f'relative_encoder_{RUN_LABEL}'
print('run存在:',trial_run_dir.exists())
for name in ['run_status.json','latest_checkpoint.json','epoch_metrics.jsonl']:
    p=trial_run_dir/name
    print(name,p.read_text()[-10000:] if p.is_file() else '未作成')
train_source=prepared_notebook['cells'][21]['source']
train_source=''.join(train_source) if isinstance(train_source,list) else train_source
print('補助セル停止原因確認:',repr(train_source[-700:]))
print('GPU空きGiB:',torch.cuda.mem_get_info()[0]/1024**3)


In [ ]:
import re, os, time
from runtime_state import sha256
anchors={}
for lp in [WORK_DIR/f'download_{original_stamp}.log',WORK_DIR/f'download_parallel_{original_stamp}.log']:
    for line in lp.read_text().splitlines():
        m=re.match(r'^\[\d+/\d+\] (\S+) (\S+) \(([^,]+), ([^)]+)\)$',line)
        if m:
            scene,filename,ih,dh=m.groups()
            for key,h in [(f'{scene}/images/{filename}',ih),(f'{scene}_depth/{Path(filename).stem}_depth.tif',dh)]:
                if re.fullmatch('[0-9a-f]{64}',h): anchors[key]=h
verified=0
for split in ['train','validation']:
    with (PLAN_DIR/f'{split}.csv').open(encoding='utf-8-sig',newline='') as f:
        for row in csv.DictReader(f):
            for local,remote,key in [(LOCAL_RGB_ROOT,DRIVE_RGB_ROOT,'image_relpath'),(LOCAL_DEPTH_ROOT,DRIVE_DEPTH_ROOT,'depth_relpath')]:
                relative=row[key]; expected=anchors.get(relative) or sha256(remote/relative)
                assert sha256(local/relative)==expected, f'コピー内容SHA256不一致: {relative}'
                verified+=1
print('ローカルSSD内容SHA256一致:',verified,'/1024',flush=True)
assert verified==1024
assert not trial_run_dir.exists(), '既存runがあるため二重起動を停止'
assert not [(p.pid) for p in psutil.process_iter(['cmdline']) if any('/train.py' in s for s in (p.info['cmdline'] or []))], '実行中の学習あり'
assert RUN_KIND=='trial' and EPOCHS==1 and TRAINING_MODES==['relative_encoder']
assert IMAGE_SIZE==518 and BATCH_SIZE==1 and GRAD_ACCUMULATION==4 and USE_AMP and SEED==42 and MAX_DEPTH_M==65.0
ACTIVE_RGB_ROOT,ACTIVE_DEPTH_ROOT=LOCAL_RGB_ROOT,LOCAL_DEPTH_ROOT
continuation_stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
training_log=WORK_DIR/f'trial_training_{continuation_stamp}.log'
trial_start=time.monotonic()
assert train_source.count('RUN_TRAINING = False')==1 and train_source.count('subprocess.run(cmd,check=True)')==1
background_call='train_process = subprocess.Popen(cmd, stdout=training_log_stream, stderr=subprocess.STDOUT, env={**os.environ,"PYTHONUNBUFFERED":"1"}); print("1エポック試験学習開始 PID:",train_process.pid,"ログ:",training_log,flush=True)'
launch_source=train_source.replace('RUN_TRAINING = False','RUN_TRAINING = True').replace('subprocess.run(cmd,check=True)',background_call)
with training_log.open('x') as training_log_stream: exec(launch_source,globals())
record={'config':TRAIN_CONFIG,'run_dir':str(trial_run_dir),'log':str(training_log),'pid':train_process.pid,'local_sha256_verified_files':verified,'preflight':str(PREFLIGHT_REPORTS['relative_encoder']),'started_utc':datetime.now(timezone.utc).isoformat(),'compute_policy':'free T4 only; no purchase'}
with (WORK_DIR/f'trial_started_{continuation_stamp}.json').open('x') as f: json.dump(record,f,indent=2)
print('学習起動記録:',record,flush=True)


In [ ]:
print('学習PID:',train_process.pid,'終了コード:',train_process.poll(),'開始から秒:',round(time.monotonic()-trial_start,1))
print('ログ末尾:',training_log.read_text()[-14000:])
for name in ['run_status.json','latest_checkpoint.json','epoch_metrics.jsonl']:
    p=trial_run_dir/name
    print(name,p.read_text()[-14000:] if p.is_file() else '未作成')
print('GPU使用:',subprocess.run(['nvidia-smi','--query-gpu=name,memory.used,memory.total,utilization.gpu','--format=csv,noheader'],capture_output=True,text=True).stdout)

p=psutil.Process(train_process.pid) if train_process.poll() is None else None
print('CPU/IO稼働:', [(q.pid,q.status(),q.cpu_times().user,q.cpu_times().system,q.io_counters().read_bytes) for q in ([p]+p.children())] if p else '終了済み')


In [ ]:
assert train_process.poll()==0, '学習がまだ実行中、または異常終了しています。再学習は開始しません。'
status=json.loads((trial_run_dir/'run_status.json').read_text())
assert status['status']=='completed' and status['completed_epochs']==1
checkpoint_verification={}
for cp in sorted(trial_run_dir.glob('checkpoint_epoch_*')):
    marker=json.loads((cp/'complete.json').read_text())
    for name,expected in marker['file_sha256'].items(): assert sha256(cp/name)==expected, f'checkpoint SHA不一致:{cp/name}'
    checkpoint_verification[cp.name]=marker
print('チェックポイント内容SHA256:',list(checkpoint_verification),'全一致')
metrics_rows=[json.loads(s) for s in (trial_run_dir/'epoch_metrics.jsonl').read_text().splitlines() if s.strip()]
print('試験学習結果:',json.dumps(metrics_rows,ensure_ascii=False,indent=2))
with (WORK_DIR/f'trial_checkpoint_verified_{continuation_stamp}.json').open('x') as f: json.dump(checkpoint_verification,f,indent=2)
print('validationの学習前評価を開始。testは使用しません。')
eval_before_output=WORK_DIR/f'relative_encoder_before_validation_{continuation_stamp}.json'
eval_before_log=WORK_DIR/f'relative_encoder_before_validation_{continuation_stamp}.log'
cmd=[sys.executable,'-u',str(CODE_DIR/'evaluate.py'),'--model',str(trial_run_dir/'checkpoint_epoch_0000'),'--manifest',str(PLAN_DIR/'validation.csv'),'--rgb-root',str(LOCAL_RGB_ROOT),'--depth-root',str(LOCAL_DEPTH_ROOT),'--image-size','518','--max-depth','65','--output',str(eval_before_output)]
with eval_before_log.open('x') as f: eval_process=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT)
print('学習前評価PID:',eval_process.pid,'結果保存先:',eval_before_output)


In [ ]:
print('学習前評価 終了コード:',eval_process.poll())
print(eval_before_log.read_text()[-2000:])
if eval_before_output.exists(): print('学習前指標:',eval_before_output.read_text())
p=psutil.Process(eval_process.pid) if eval_process.poll() is None else None
print('評価CPU/IO:',[(q.pid,q.status(),q.cpu_times().user,q.io_counters().read_bytes) for q in ([p]+p.children())] if p else '終了済み')


In [ ]:
assert eval_process.poll()==0 and eval_before_output.is_file(), '学習前評価がまだ完了していません'
before=json.loads(eval_before_output.read_text())
assert before['pairs']==128 and before['split']=='validation'
print('Relative初期モデルのvalidation:',json.dumps(before['metrics'],indent=2))
metric_cached=Path(json.loads((WORK_DIR/f'model_paths_both_{original_stamp}.json').read_text())['metric'])
assert (metric_cached/'model.safetensors').is_file() and (metric_cached/'config.json').is_file()
canonical_cap=float(json.loads((metric_cached/'config.json').read_text())['max_depth'])
assert canonical_cap==80.0, '比較モデルの元設定が想定外です'
baseline_output=WORK_DIR/f'metric_outdoor_original_validation_{continuation_stamp}.json'
baseline_log=WORK_DIR/f'metric_outdoor_original_validation_{continuation_stamp}.log'
baseline_started=time.monotonic()
cmd=[sys.executable,'-u',str(CODE_DIR/'evaluate.py'),'--model',str(metric_cached),'--manifest',str(PLAN_DIR/'validation.csv'),'--rgb-root',str(LOCAL_RGB_ROOT),'--depth-root',str(LOCAL_DEPTH_ROOT),'--image-size','518','--max-depth',str(canonical_cap),'--output',str(baseline_output)]
with baseline_log.open('x') as f: baseline_process=subprocess.Popen(cmd,stdout=f,stderr=subprocess.STDOUT)
print('既存Metric Outdoor Smallのみを評価。再ダウンロード・追加学習なし。PID:',baseline_process.pid)


In [ ]:
print('Metric Outdoor評価 終了コード:',baseline_process.poll(),'経過秒:',round(time.monotonic()-baseline_started,1))
print(baseline_log.read_text()[-1000:])
if baseline_output.exists(): print('既存Metric Outdoor指標:',baseline_output.read_text())
p=psutil.Process(baseline_process.pid) if baseline_process.poll() is None else None
print('CPU/IO:',[(q.pid,q.status(),q.cpu_times().user,q.io_counters().read_bytes) for q in ([p]+p.children())] if p else '終了済み')


In [ ]:
assert baseline_process.poll()==0 and eval_process.poll()==0 and train_process.poll()==0
baseline=json.loads(baseline_output.read_text()); before=json.loads(eval_before_output.read_text())
after=metrics_rows[-1]
assert baseline['pairs']==before['pairs']==128
assert baseline['manifest_sha256']==before['manifest_sha256']==sha256(PLAN_DIR/'validation.csv')
assert baseline['metrics']['valid_pixels']==before['metrics']['valid_pixels']==after['validation']['valid_pixels']
assert baseline['metrics']['local_pair_count']==after['validation']['local_pair_count']
ssd=dataset_status(PLAN_DIR,LOCAL_RGB_ROOT,LOCAL_DEPTH_ROOT,['train','validation'])
persistent=dataset_status(PLAN_DIR,DRIVE_RGB_ROOT,DRIVE_DEPTH_ROOT,['train','validation'])
assert ssd['complete'] and persistent['complete']
summary={'completed_utc':datetime.now(timezone.utc).isoformat(),'config':TRAIN_CONFIG,'ssd':ssd,'drive':persistent,'copied_files_sha256_verified':1024,'training_returncode':0,'run_dir':str(trial_run_dir),'run_status':json.loads((trial_run_dir/'run_status.json').read_text()),'epoch':after,'relative_initial_validation':before,'metric_outdoor_original_validation':baseline,'preflight':json.loads(PREFLIGHT_REPORTS['relative_encoder'].read_text()),'checkpoint_integrity':checkpoint_verification,'gpu_no_longer_needed':True,'compute_units':'Colab UI: no available Compute Units; free GPU; numerical consumption unavailable; no purchases','test_used':False,'no_redownload_this_continuation':True,'estimate_20_epochs_seconds_excludes_save_and_startup':after['epoch_seconds']*20}
summary_path=WORK_DIR/f'execution_summary_{continuation_stamp}.json'
with summary_path.open('x') as f: json.dump(summary,f,ensure_ascii=False,indent=2,allow_nan=False)
print('COMPARISON_SUMMARY_JSON',json.dumps(summary,ensure_ascii=False))
print('永続保存:',summary_path)
print('GPUプロセス:',subprocess.run(['nvidia-smi','--query-compute-apps=pid,process_name,used_gpu_memory','--format=csv,noheader'],capture_output=True,text=True).stdout)
assert not [p.pid for p in psutil.process_iter(['cmdline']) if any('/train.py' in s or '/evaluate.py' in s or 'stage_selected_data.py' in s for s in (p.info['cmdline'] or []))]
print('学習・評価・コピーの実行中プロセスはありません。GPU終了可能。')


In [ ]:
report_text="# Colab実行報告（2026-10-13）\n\n## 再開時の確認\n前回のローカルSSDコピーは正常終了（終了コード0）。学習は補助セルの文字列一致検査で停止し、未開始だった。停止原因は subprocess.run(cmd,check=True) の空白差。補助セルのみ修正し、研究コード・分割計画は変更していない。\nDriveとSSDの両方で512組・1024ファイル、7,850,330,776 bytes、欠損0・サイズ不一致0。SSD全1024ファイルをSHA256で確認。1023ファイルはCPU取得時ログ、残り1ファイルはDrive原本とのSHA256比較で一致。再取得・再コピー・画素再監査は実施していない。\n\n## 保存先と準備\nMyDrive/DepthAnythingV2-UAV/uav_metric_finetuning にコード・Notebook・config/verified_planを配置済み。RGBは data/uav3dcrop_rgb、GTは data/uav3dcrop_depth。作業レポートは work/uav_metric_finetuning。既存成果物は保持した。\ntrain384（Day019/028/040）、validation128（Day047）。独立test319は未取得・未使用。Day001/007と旧52組のシーンは隔離。\nCPU監査pass：RGB/GT同解像度3956×5280、GT float16、NaN/Inf/負値0。無効GTは0。全512組の有効画素10,671,514,368、無効ゼロ22,977,792。\nGPU切替時のDrive更新時刻丸め77件はCPU取得ログのSHA256との一致で再検証し、旧レポートを保持して派生監査・GTレポートを作成。監査コード・仕様・manifest・保存先・サイズ・環境の一致を確認して再利用した。\ntrainのGTのみ解析：p99.9=52.90625m、全有効画素最大62.15625m。60m超15,424画素（0.0001927%）。全train GTを包含するためmax_depth=65mを選択。validation/testによる上限選択なし。\n\n## GPUと試験条件\n無料Tesla T4、総VRAM14.563GiB。CUDA/PyTorch GPU利用可。\n固定revisionのRelative SmallとMetric Outdoor Smallのcacheを再利用。今回relative_encoder方式（エンコーダ転用、新Metric head）、AdamW、SiLog、518×518、batch1、蓄積4、AMP、seed42、1epoch、max_depth65m。\n事前計測pass：Forward/Backward・AdamW step成功、数値検査pass。allocated624,851,456B、reserved815,792,128B、合成step22.656秒。実データ全体の成功をこの事前計測だけで保証していない。\n\n## 実測結果\n全モデルを同じvalidation128組・元解像度で評価。有効画素2,668,757,375、局所差ペア5,320,891,492で一致。局所差は14pixel間隔。深度はm、AbsRelと相関は無次元。\n\n|指標|Relative encoder＋新head（学習前）|1epoch学習後|既存Metric Outdoor Small|\n|---|---:|---:|---:|\n|MAE m|13.8251|2.4897|13.2730|\n|RMSE m|14.7163|3.9063|14.5340|\n|AbsRel|0.8324|0.1212|0.6590|\n|局所深度差MAE m|0.06437|0.10545|0.06561|\n|Pearson r|-0.05599|0.87117|0.78475|\n\n学習前はRelativeモデルの直接的なMetric評価ではなく、転用encoder＋新Metric headのcheckpoint_epoch_0000。既存Metric Outdoorは元の上限80mを維持（65mへ変更せず）、追加学習なし。有効画素は全モデルで同じ。\nSiLog Loss=0.29133354。学習＋validation=774.469秒（12分54秒、checkpoint保存・起動は別）。実測VRAM peak allocated943,323,648B（0.879GiB）、reserved1,340,080,128B（1.248GiB）。学習正常終了コード0、completed1、remaining0。\n作物別RMSE：corn4.9581m、oat3.4187m、soybean3.6021m、wheat3.4340m。\n平均深度誤差・相関は既存モデルより改善。しかし局所深度差は悪化し、独立評価は未実施。稲のGT・高さ評価も未実施。この結果で稲の高さ精度の改善を主張できない。\n\n## checkpointと資源管理\nruns/relative_encoder_trial_20261013_s42_518/checkpoint_epoch_0000 および checkpoint_epoch_0001 をDriveへ保存。complete.jsonの全保存ファイルSHA256が一致。モデル読み込みによる学習前評価も成功。optimizer/scheduler/scaler状態も保存され、完全性は確認済み。今回のGPU中断再開は実施していない。\n約20分07秒のSSDコピーは前回に完了。データ取得は前回約11分26秒。今回の追加処理は試験学習＋学習前評価＋既存モデル評価。既存モデル評価は約9分39秒の完了確認時経過（監視間隔を含む）。\n20epoch単純概算258.16分（約4時間18分、起動・保存除外）。無料GPUの継続時間・同GPU割当は保証されない。Compute Units表示は利用可能ユニットなしで、実消費量の数値を取得できない。追加購入・課金・プラン変更なし。\nGPUを使う処理はすべて終了。Drive保存確認後にランタイムを終了する。終了確認は最終メッセージを参照。\n\n## 次の作業\n本学習は新runで開始する。試験1epochから20epochへの強制resumeは行わない。本学習中のresumeは同じ総epoch数・設定・コード・データ・パッケージ条件を保持する。\n518pxのメモリは今回T4で十分だが、次回GPU割当では再計測する。GT65m、SiLog、seed42を維持し、まずmetric_outdoor初期化方式の比較実験と局所形状・稲/地面のGT評価を計画する。損失の無断変更なし。独立testはモデル選択完了後まで保持する。\nセッション終了でローカルSSDは失われる。次回はDrive原本から必要時のみSSDへコピーし、HFからの再取得は不要。0.初期化→環境→設定/監査再利用→モデルcache→新GPU事前計測の順で復元する。\n\n実行Notebook：https://colab.research.google.com/drive/1_kykQhNEVXorSIVAsl_dKYxqpJZDcbmk\n詳細JSON：work/uav_metric_finetuning/execution_summary_20261013T113119407440Z.json\n"
report_path=WORK_DIR/f'execution_report_ja_{continuation_stamp}.md'
with report_path.open('x',encoding='utf-8') as f: f.write(report_text)
print('実行報告書保存:',report_path)
print('512組/1024ファイル: SSD・Drive整合性PASS、コピーSHA256全一致')
print('1epoch試験学習: PASS; SiLog=0.29133; 12分54秒')
print('validation比較（MAE / RMSE / AbsRel / 局所差MAE / Pearson）')
for label,m in [('学習前',before['metrics']),('学習後',after['validation']),('既存Metric Outdoor',baseline['metrics'])]:
    print(label,*(round(m[k],5) for k in ['mae_m','rmse_m','abs_rel','local_depth_delta_mae_m','pearson_r']))
print('checkpoint_epoch_0000/0001: 保存ファイルSHA256 PASS')
print('実行中学習・評価プロセスなし。Drive同期後にGPUを終了します。')


In [ ]:
assert train_process.poll()==0 and eval_process.poll()==0 and baseline_process.poll()==0
assert summary_path.is_file() and report_path.is_file()
assert json.loads((trial_run_dir/'run_status.json').read_text())['status']=='completed'
print('Drive成果物確認: データ・実行JSON・日本語報告・checkpoint保存済み')
from google.colab import drive
drive.flush_and_unmount()
print('Drive同期・アンマウント完了。GPUランタイム終了可能。')


## 9. GPU: CUDA・メモリ事前計測

ここからはGPUを使用します。CUDAが使えるColab runtimeを選び、このセルを個別に実行してください。合成1 stepのみで、UAVデータは読み込みません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_GPU_PREFLIGHT = False
PREFLIGHT_REPORTS = {}
if RUN_GPU_PREFLIGHT:
    import torch
    refresh_preparation_state()
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable。Colab runtimeをGPUに変更してください。CPU準備は完了しています。')
    if MAX_DEPTH_M is None or not AUDIT_OK or GT_REPORT is None: raise RuntimeError('GT max_depth設定とデータ監査passが必要です。')
    if not all(('relative' if m=='relative_encoder' else 'metric') in MODEL_PATHS for m in TRAINING_MODES):
        raise RuntimeError('選択方式のモデル重みがありません。前のモデルキャッシュセルを確認してください。')
    print('GPU:',torch.cuda.get_device_name(0),'| total GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),
          '| free GiB:',round(torch.cuda.mem_get_info()[0]/1024**3,2),
          '| /content free GiB:',round(shutil.disk_usage('/content').free/1024**3,2))
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    relative_path=MODEL_PATHS.get('relative',RELATIVE_MODEL_ID)
    metric_path=MODEL_PATHS.get('metric',METRIC_OUTDOOR_MODEL_ID)
    for mode in TRAINING_MODES:
        report=PREFLIGHT_DIR/f'{mode}_res{IMAGE_SIZE}_bs{BATCH_SIZE}_{stamp}.json'
        cmd=[sys.executable,str(CODE_DIR/'memory_preflight.py'),'--initialization',mode,
             '--relative-model',relative_path,'--metric-model',metric_path,'--image-size',str(IMAGE_SIZE),
             '--batch-size',str(BATCH_SIZE),'--max-depth',str(MAX_DEPTH_M),'--output',str(report),
             '--amp' if USE_AMP else '--no-amp']
        subprocess.run(cmd,check=True)
        PREFLIGHT_REPORTS[mode]=report
else: print('RUN_GPU_PREFLIGHT=False: GPUを使いません。')

## 10. GPU: 試験学習・本学習

まずEPOCHS=1の短い試験runを推奨します。学習前にmanifest、ファイル存在、GT設定、モデル、同一設定の事前計測が必須です。1epochごとにDriveへチェックポイントを保存し、時間見積もりと残りepochを表示します。testは読み込みません。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_TRAINING = False
if RUN_TRAINING:
    import torch, hashlib
    refresh_preparation_state()
    if not torch.cuda.is_available(): raise RuntimeError('このセルはCUDAが必要です。')
    if not AUDIT_OK or GT_REPORT is None or MAX_DEPTH_M is None: raise RuntimeError('audit passとtrain-only GT/max_depthが必要です。')
    train_manifest_sha=hashlib.sha256((PLAN_DIR/'train.csv').read_bytes()).hexdigest()
    if GT_REPORT.get('source_manifest_sha256')!=train_manifest_sha: raise RuntimeError('GT解析とtrain manifestが一致しません。GTを再解析してください。')
    if AUDIT_REPORT.get('manifest_sha256',{}).get('train')!=train_manifest_sha or AUDIT_REPORT.get('manifest_sha256',{}).get('validation')!=hashlib.sha256((PLAN_DIR/'validation.csv').read_bytes()).hexdigest(): raise RuntimeError('監査対象manifestと学習manifestが一致しません。監査を再実行してください。')
    if set(PREFLIGHT_REPORTS)!=set(TRAINING_MODES): raise RuntimeError('各選択方式のGPU事前計測を先にpassさせてください。')
    if not RUN_LABEL or '/' in RUN_LABEL or chr(92) in RUN_LABEL: raise ValueError('RUN_LABELを単一フォルダー名にしてください。')
    preparation_context(manifests={'train':PLAN_DIR/'train.csv','validation':PLAN_DIR/'validation.csv'}, rgb_root=ACTIVE_RGB_ROOT, depth_root=ACTIVE_DEPTH_ROOT, physical_splits=['train','validation'])
    train_count=plan_summary['selected_pair_counts']['train']
    step_count=(train_count+BATCH_SIZE-1)//BATCH_SIZE
    print(f'{train_count} pairs/epoch; optimizer steps/epoch={(step_count+GRAD_ACCUMULATION-1)//GRAD_ACCUMULATION}; epochs={EPOCHS}')
    print('初回epoch終了後、実測時間から総所要時間と残り時間を概算します。')
    relative_path=MODEL_PATHS.get('relative',RELATIVE_MODEL_ID)
    metric_path=MODEL_PATHS.get('metric',METRIC_OUTDOOR_MODEL_ID)
    for mode in TRAINING_MODES:
        run_dir=RUNS_DIR/f'{mode}_{RUN_LABEL}'
        cmd=[sys.executable,str(CODE_DIR/'train.py'),'--train-manifest',str(PLAN_DIR/'train.csv'),
             '--val-manifest',str(PLAN_DIR/'validation.csv'),'--rgb-root',str(ACTIVE_RGB_ROOT),
             '--depth-root',str(ACTIVE_DEPTH_ROOT),'--initialization',mode,'--relative-model',relative_path,
             '--metric-model',metric_path,'--max-depth',str(MAX_DEPTH_M),'--image-size',str(IMAGE_SIZE),
             '--batch-size',str(BATCH_SIZE),'--grad-accumulation',str(GRAD_ACCUMULATION),
             '--epochs',str(EPOCHS),'--run-kind',RUN_KIND,'--keep-recent-checkpoints','0','--workers',str(WORKERS),'--seed',str(SEED),
             '--save-every-epochs','1','--memory-preflight-report',str(PREFLIGHT_REPORTS[mode]),
             '--output-dir',str(run_dir),'--amp' if USE_AMP else '--no-amp']
        if RESUME_TRAINING: cmd.append('--resume')
        subprocess.run(cmd,check=True)
else: print('RUN_TRAINING=False: 学習を開始していません。')

## 10b. CPUでも確認可能: 試験結果・checkpoint・時間概算

このセルは学習を開始しません。試験後に損失、validation各指標、時間、実測VRAMを確認してください。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('最初のDrive接続・状態復元セルを実行してください。')
ESTIMATE_EPOCHS = 20  # 時間の概算だけ。学習は開始しません。
for mode in TRAINING_MODES:
    run_dir = RUNS_DIR/f'{mode}_{RUN_LABEL}'
    log = run_dir/'epoch_metrics.jsonl'
    if not log.is_file():
        print('学習結果はまだありません:',run_dir)
        continue
    rows = [json.loads(line) for line in log.read_text(encoding='utf-8').splitlines() if line.strip()]
    row = rows[-1]
    print(mode, '| epoch:',row['epoch'],'| SiLog:',row['train_silog'])
    print('validation:',row['validation'])
    print('epoch秒:',row['epoch_seconds'],'| peak allocated/reserved GiB:',
          row.get('peak_allocated_bytes',0)/1024**3, row.get('peak_reserved_bytes',0)/1024**3)
    mean = sum(r['epoch_seconds'] for r in rows)/len(rows)
    print(f'{ESTIMATE_EPOCHS}epochの概算: {mean*ESTIMATE_EPOCHS/60:.1f}分（checkpoint保存・接続待ち等は別。CCU消費量ではありません）')
    print('run状態:',(run_dir/'run_status.json').read_text(encoding='utf-8'))
    print('checkpoint:',run_dir/'latest_checkpoint.json',run_dir/'best_checkpoint.json')


## 11. 最終段階: 独立test（1回）

モデル・設定を確定し、independent_testを別途取得・監査した後だけ実行します。testを見てモデルを選び直した場合、そのtestは独立評価ではなくなります。

In [ ]:
if not globals().get('SESSION_READY', False):
    raise RuntimeError('実行状態がありません。最初に「0. Drive接続・状態復元」の初期化セルを実行してください。GPU切替後にも必要です。')

RUN_INDEPENDENT_TEST = False
TEST_MODE = 'relative_encoder'
if RUN_INDEPENDENT_TEST:
    if MAX_DEPTH_M is None: raise RuntimeError('先にtrain-only GT解析と学習設定を確認してください。')
    if not (RUNS_DIR/f'{TEST_MODE}_{RUN_LABEL}'/'best_checkpoint.json').is_file(): raise RuntimeError('最終モデルがありません。試験結果・validationによるモデル選択を完了してから独立testを実行してください。')
    import hashlib
    if TEST_MODE not in TRAINING_MODES: raise ValueError('TEST_MODEは学習済み方式にしてください。')
    run_dir=RUNS_DIR/f'{TEST_MODE}_{RUN_LABEL}'
    best=json.loads((run_dir/'best_checkpoint.json').read_text(encoding='utf-8'))
    checkpoint=run_dir/best['path']
    test_audit=WORK_DIR/f'test_audit_{datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")}.json'
    subprocess.run([sys.executable,str(CODE_DIR/'audit_data.py'),'--train',str(PLAN_DIR/'train.csv'),
        '--validation',str(PLAN_DIR/'validation.csv'),'--independent-test',str(PLAN_DIR/'independent_test.csv'),
        '--physical-splits','train','validation','independent_test','--plan-summary',str(PLAN_DIR/'plan_summary.json'),
        '--rgb-root',str(DRIVE_RGB_ROOT),'--depth-root',str(DRIVE_DEPTH_ROOT),'--output',str(test_audit)],check=True)
    if json.loads(test_audit.read_text(encoding='utf-8')).get('status')!='pass': raise RuntimeError('testを含む物理監査がpassしていません。')
    output=WORK_DIR/f'{TEST_MODE}_{RUN_LABEL}_independent_test.json'
    subprocess.run([sys.executable,str(CODE_DIR/'evaluate.py'),'--model',str(checkpoint),
        '--manifest',str(PLAN_DIR/'independent_test.csv'),'--rgb-root',str(DRIVE_RGB_ROOT),
        '--depth-root',str(DRIVE_DEPTH_ROOT),'--image-size',str(IMAGE_SIZE),
        '--max-depth',str(MAX_DEPTH_M),'--output',str(output)],check=True)
else: print('RUN_INDEPENDENT_TEST=False: testは実行していません。')